# Main Performance Experiments

This notebook is the public, output-free entry point for reproducing the paper experiments.
Datasets and generated result files are intentionally excluded from the repository.


## Full main performance workflow


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()

def _looks_like_project_root(path):
    markers = ["model.py", "data.py", "learning.py", "utils.py"]
    return all((path / marker).exists() for marker in markers)

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E316",
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_exp",
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)

os.environ.setdefault("OPENML_DATA_HOME", str(PROJECT_ROOT / "data" / "openml_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
(PROJECT_ROOT / "data" / "openml_cache").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS, TSKANFIS, ParallelHierarchicalTSKANFIS
from utils import set_deterministic, build_loader, load_gh_params, load_ga_params, load_pso_params
from data import *
from learning import *
from gh_eval_utils import *
from interpretability import nauck_index_gh, nauck_index_tsk, nauck_index_parallel_hier_tsk
from sklearn.svm import SVC
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from tqdm import tqdm

print(f"Project root: {PROJECT_ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

set_deterministic(SEED)


In [ ]:
def _metric_average(task_kind):
    return 'weighted' if task_kind == 'multiclass' else 'binary'


def _compute_cls_metrics(y_true, y_pred, task_kind):
    average = _metric_average(task_kind)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
        'recall': recall_score(y_true, y_pred, average=average, zero_division=0),
        'f1': f1_score(y_true, y_pred, average=average, zero_division=0),
    }


def _eval_torch_cls(model, loader, task_kind):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device).float()
            y_batch = y_batch.to(device)
            logits = model(x_batch)

            if task_kind == 'binary':
                if logits.dim() == 1:
                    logits = logits.unsqueeze(1)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long().squeeze(1)
                targets = y_batch.long().squeeze(1) if y_batch.dim() > 1 else y_batch.long()
            else:
                preds = torch.argmax(logits, dim=1)
                targets = y_batch.long()

            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_targets, axis=0)
    return _compute_cls_metrics(y_true, y_pred, task_kind)


def _eval_sklearn_cls(model, X, y, task_kind):
    preds = model.predict(X)
    return _compute_cls_metrics(y, preds, task_kind)


def _eval_gh_detailed_cls(model, loader, task_kind):
    results = {}

    for mode_name in ['base_only', 'residual_only', 'full']:
        model.set_mode(mode_name)
        model.eval()

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for x_batch, y_batch in loader:
                x_batch = x_batch.to(device).float()
                y_batch = y_batch.to(device)
                logits = model(x_batch)

                if task_kind == 'binary':
                    if logits.dim() == 1:
                        logits = logits.unsqueeze(1)
                    probs = torch.sigmoid(logits)
                    preds = (probs >= 0.5).long().squeeze(1)
                    targets = y_batch.long().squeeze(1) if y_batch.dim() > 1 else y_batch.long()
                else:
                    preds = torch.argmax(logits, dim=1)
                    targets = y_batch.long()

                all_preds.append(preds.cpu().numpy())
                all_targets.append(targets.cpu().numpy())

        y_pred = np.concatenate(all_preds, axis=0)
        y_true = np.concatenate(all_targets, axis=0)
        key = 'combined' if mode_name == 'full' else mode_name.replace('_only', '')
        results[key] = _compute_cls_metrics(y_true, y_pred, task_kind)

    model.set_mode('full')
    return results



from pathlib import Path
import copy
import joblib

CV_WEIGHT_ROOT = Path("./hyper_parameter/cv_weights")
_MODEL_FILE_STEM = {
    "GH-ANFIS": "gh_anfis",
    "ANFIS": "anfis",
    "GA-ANFIS": "ga_anfis",
    "PSO-ANFIS": "pso_anfis",
    "PH-ANFIS(Avg)": "ph_anfis_avg",
    "PH-ANFIS(Stacked)": "ph_anfis_stacked",
    "SVM": "svm",
}


def _safe_name(name):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(name))


def _scaler_stats(scaler):
    if scaler is None:
        return None, None
    mean = getattr(scaler, "mean_", None)
    scale = getattr(scaler, "scale_", None)
    return (mean.tolist() if mean is not None else None, scale.tolist() if scale is not None else None)


def save_cv_torch_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    params,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    scaler_mean, scaler_scale = _scaler_stats(scaler)
    stem = _MODEL_FILE_STEM[model_name]
    out_path = fold_dir / f"{stem}.pt"

    state_dict_cpu = {
        k: (v.detach().cpu() if torch.is_tensor(v) else v)
        for k, v in model.state_dict().items()
    }

    payload = {
        "framework": "torch",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "params": copy.deepcopy(params),
        "scaler_mean": scaler_mean,
        "scaler_scale": scaler_scale,
        "state_dict": state_dict_cpu,
        "extra_meta": dict(extra_meta or {}),
    }
    torch.save(payload, out_path)
    return str(out_path)


def save_cv_sklearn_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    stem = _MODEL_FILE_STEM[model_name]
    out_path = fold_dir / f"{stem}.joblib"

    payload = {
        "framework": "sklearn",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "scaler": scaler,
        "model": model,
        "extra_meta": dict(extra_meta or {}),
    }
    joblib.dump(payload, out_path)
    return str(out_path)


def load_cv_artifact(dataset_name, model_name, fold_idx, root_dir=CV_WEIGHT_ROOT, device=device):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    stem = _MODEL_FILE_STEM[model_name]

    if model_name == "SVM":
        payload = joblib.load(fold_dir / f"{stem}.joblib")
        model = payload["model"]
        return model, payload

    payload = torch.load(fold_dir / f"{stem}.pt", map_location=device)
    params = payload.get("params") or {}
    n_features = int(payload["n_features"])
    n_outputs = int(payload["n_outputs"])

    if model_name == "GH-ANFIS":
        model = GH_ANFIS(
            n_features=n_features,
            n_outputs=n_outputs,
            residual_rules=int(params.get("residual_rules", 8)),
            base_rules=int(params.get("base_rules", 4)),
            mf_per_feature=int(params.get("mf_per_feature", 2)),
            device=device,
        ).to(device)
    elif model_name in ("PH-ANFIS(Avg)", "PH-ANFIS(Stacked)"):
        fusion_default = "avg" if model_name == "PH-ANFIS(Avg)" else "stacked"
        fusion = str(params.get("fusion", fusion_default))
        extra_meta = payload.get("extra_meta") or {}
        group_a_idx = list(extra_meta.get("group_a_idx") or [])
        group_b_idx = list(extra_meta.get("group_b_idx") or [])
        if not group_a_idx or not group_b_idx:
            split_seed = int(params.get("split_seed", SEED))
            rng = np.random.default_rng(split_seed)
            order = np.arange(int(n_features), dtype=int)
            rng.shuffle(order)
            cut = int(max(1, n_features // 2))
            if cut >= n_features:
                cut = n_features - 1
            group_a_idx = np.sort(order[:cut]).tolist()
            group_b_idx = np.sort(order[cut:]).tolist()

        model = ParallelHierarchicalTSKANFIS(
            n_inputs=n_features,
            n_outputs=n_outputs,
            group_a_idx=group_a_idx,
            group_b_idx=group_b_idx,
            branch_rules=int(params.get("branch_rules", params.get("n_rules", 12))),
            top_rules=int(params.get("top_rules", max(2, int(params.get("branch_rules", params.get("n_rules", 12))) // 2))),
            fusion=fusion,
            mfs_per_input=int(params.get("mfs_per_input", 3)),
        ).to(device)
    else:
        model = TSKANFIS(
            n_inputs=n_features,
            n_rules=int(params.get("n_rules", 30)),
            n_outputs=n_outputs,
            mfs_per_input=int(params.get("mfs_per_input", 3)),
        ).to(device)

    model.load_state_dict(payload["state_dict"])
    model.eval()
    return model, payload


def build_scaler_from_artifact(payload):
    mean = payload.get("scaler_mean")
    scale = payload.get("scaler_scale")
    if mean is None or scale is None:
        return None
    scaler = StandardScaler()
    scaler.mean_ = np.asarray(mean, dtype=float)
    scaler.scale_ = np.asarray(scale, dtype=float)
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = int(scaler.mean_.shape[0])
    scaler.n_samples_seen_ = 1
    return scaler


def _normalize_feature_key(name):
    text = str(name).strip()
    return "_".join(part for part in ''.join(ch if ch.isalnum() else '_' for ch in text).split('_') if part).lower()


def select_features_df(X, selected, label):
    if not selected:
        return X, list(X.columns) if hasattr(X, "columns") else None
    if not hasattr(X, "columns"):
        return X, None

    available_cols = list(X.columns)
    exact_map = {str(col): col for col in available_cols}
    normalized_map = {}
    for col in available_cols:
        normalized_map.setdefault(_normalize_feature_key(col), []).append(col)

    resolved = []
    missing = []

    for feat in selected:
        feat_str = str(feat)
        if feat_str in exact_map:
            resolved.append(exact_map[feat_str])
            continue

        if feat_str.startswith("x") and feat_str[1:].isdigit():
            idx = int(feat_str[1:])
            if idx < X.shape[1]:
                resolved.append(X.columns[idx])
                continue

        candidates = normalized_map.get(_normalize_feature_key(feat_str), [])
        if len(candidates) == 1:
            resolved.append(candidates[0])
            continue

        missing.append(feat)

    resolved = list(dict.fromkeys(resolved))
    if resolved:
        if missing:
            print(f"[{label}] matched {len(resolved)} selected_features; skipped {len(missing)} unmatched entries")
        return X.loc[:, resolved], list(resolved)

    print(f"[{label}] selected_features not found; using all features")
    return X, list(X.columns)


PH_PARAMS_BY_DATASET = {
    "Vowel": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 18,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Breast_Cancer_Wisconsin_(Original)": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 16,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Spambase": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 20,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Gisette": {
        "lr": 0.03,
        "epochs": 70,
        "branch_rules": 14,
        "top_rules": 6,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
}

PH_VARIANTS = [
    ("PH-ANFIS(Avg)", "avg", 101),
    ("PH-ANFIS(Stacked)", "stacked", 211),
]


In [ ]:
dataset_list = ['Breast_Cancer_Wisconsin_(Original)',
                'Vowel',
                'Spambase',
                'Gisette']
model_list = ['GH-ANFIS', 'ANFIS', 'GA-ANFIS', 'PSO-ANFIS', 'PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)', 'SVM']

# Vowel
Vowel_x, Vowel_y, Vowel_feature_name = load_vowel_data()
Vowel_x = coerce_numeric_frame(Vowel_x)
Vowel_x, Vowel_y = drop_nan_targets(Vowel_x, Vowel_y)

# Spambase
Spambase_x, Spambase_y, Spambase_feature_name = load_spambase_data()
Spambase_x = coerce_numeric_frame(Spambase_x)
Spambase_x, Spambase_y = drop_nan_targets(Spambase_x, Spambase_y)

# Gisette
Gisette_x, Gisette_y, Gisette_feature_name = load_gisette_data()
Gisette_x = coerce_numeric_frame(Gisette_x)
Gisette_x, Gisette_y = drop_nan_targets(Gisette_x, Gisette_y)

# Breasts Cancer Wisconsin (Diagnostic)
BCWD_x, BCWD_y, BCWD_feature_name = load_bcwd_data()
BCWD_x = coerce_numeric_frame(BCWD_x)
BCWD_x, BCWD_y = drop_nan_targets(BCWD_x, BCWD_y)


In [ ]:
print(Vowel_x.shape)
print(Spambase_x.shape)
print(Gisette_x.shape)
print(BCWD_x.shape)

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd


def _top_mi_columns(X_train_df, y_train, top_k, random_state=SEED):
    if top_k is None:
        return list(X_train_df.columns)
    k = int(max(1, min(int(top_k), X_train_df.shape[1])))
    if k >= X_train_df.shape[1]:
        return list(X_train_df.columns)

    mi = mutual_info_classif(
        X_train_df,
        np.asarray(y_train).astype(int),
        random_state=int(random_state),
    )
    order = np.argsort(mi)[::-1]
    return [X_train_df.columns[i] for i in order[:k]]


def _dataset_bundle(data_name):
    if data_name == 'Vowel':
        return Vowel_x, Vowel_y, Vowel_feature_name
    if data_name == 'Breast_Cancer_Wisconsin_(Original)':
        return BCWD_x, BCWD_y, BCWD_feature_name
    if data_name == 'Spambase':
        return Spambase_x, Spambase_y, Spambase_feature_name
    if data_name == 'Gisette':
        return Gisette_x, Gisette_y, Gisette_feature_name
    raise ValueError(f'Unknown dataset: {data_name}')


def _batch_size_for_dataset(data_name):
    return 4000 if data_name == 'Spambase' else 1024


def run_cv_dataset_mode(data_name, gh_params, mi_top_k=None, mode_label='no_mi'):
    X_df, y, feature_name = _dataset_bundle(data_name)

    n_classes = len(np.unique(y))
    task = 'binary' if n_classes == 2 else 'multiclass'
    n_out = 1 if task == 'binary' else n_classes
    n_folds = 5
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    ga_params = load_ga_params(data_name, data_name)
    pso_params = load_pso_params(data_name, data_name)
    tsk_params = {'lr': 0.1, 'n_rules': 30, 'epochs': 100, 'selected_features': []}
    ph_params = copy.deepcopy(PH_PARAMS_BY_DATASET[data_name])
    svm_params = {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}

    ga_selected = list(ga_params.get('selected_features') or [])
    pso_selected = list(pso_params.get('selected_features') or [])

    if data_name == 'Gisette' and mi_top_k is not None and mi_top_k > 0:
        ga_selected = ga_selected[:int(mi_top_k)]
        pso_selected = pso_selected[:int(mi_top_k)]

    print(f"[{data_name}/{mode_label}] ANFIS selected=core, GA selected={len(ga_selected)}, PSO selected={len(pso_selected)}")

    summary_models = [
        'GH-ANFIS(base)',
        'GH-ANFIS(residual)',
        'GH-ANFIS(full)',
        'ANFIS',
        'GA-ANFIS',
        'PSO-ANFIS',
        'PH-ANFIS(Avg)',
        'PH-ANFIS(Stacked)',
        'SVM',
    ]

    metric_names = ('acc', 'precision', 'recall', 'f1')
    cv_results = {name: {metric: [] for metric in metric_names} for name in summary_models}
    cv_nauck = {
        name: {'index': [], 'comp': [], 'cov': [], 'part': [], 'n_rules': [], 'n_features': []}
        for name in summary_models
    }

    def _push_nauck(name, info):
        cv_nauck[name]['index'].append(info.get('index', np.nan))
        cv_nauck[name]['comp'].append(info.get('comp', np.nan))
        cv_nauck[name]['cov'].append(info.get('cov', np.nan))
        cv_nauck[name]['part'].append(info.get('part', np.nan))
        cv_nauck[name]['n_rules'].append(info.get('n_rules', np.nan))
        cv_nauck[name]['n_features'].append(info.get('n_features', np.nan))

    def _push_metrics(name, metrics):
        for metric in metric_names:
            cv_results[name][metric].append(metrics.get(metric, np.nan))

    saved_cv_artifacts = []

    X_train_df, X_test_df, y_train_tmp, y_test_tmp = train_test_split(
        X_df, y, test_size=0.2, random_state=SEED
    )
    scaler_tmp = StandardScaler().fit(X_train_df)
    X_train_tmp = scaler_tmp.transform(X_train_df)
    X_test_tmp = scaler_tmp.transform(X_test_df)
    print(f"[{data_name}/{mode_label}] Train: {X_train_tmp.shape}, Test: {X_test_tmp.shape}")

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_df, y), 1):
        set_deterministic(SEED)
        print(f"{'='*60}")
        print(f"[{data_name}/{mode_label}] Fold {fold_idx}/{n_folds}")
        print(f"{'='*60}")

        X_train_full, X_val_full = X_df.iloc[train_idx], X_df.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if mi_top_k is None:
            X_train_core = X_train_full
            X_val_core = X_val_full
        else:
            mi_cols = _top_mi_columns(X_train_full, y_train, mi_top_k, random_state=SEED)
            X_train_core = X_train_full.loc[:, mi_cols]
            X_val_core = X_val_full.loc[:, mi_cols]
            if fold_idx == 1:
                print(f"[{data_name}/{mode_label}] MI selected={len(mi_cols)}")

        batch_size = _batch_size_for_dataset(data_name)

        scaler_core = StandardScaler().fit(X_train_core)
        X_train_core_scaled = scaler_core.transform(X_train_core)
        X_val_core_scaled = scaler_core.transform(X_val_core)
        train_loader_core = build_loader(X_train_core_scaled, y_train, batch_size, device, task)
        val_loader_core = build_loader(X_val_core_scaled, y_val, batch_size, device, task)

        X_train_tsk = X_train_core
        X_val_tsk = X_val_core
        tsk_used = list(X_train_core.columns)
        scaler_tsk = StandardScaler().fit(X_train_tsk)
        X_train_tsk_scaled = scaler_tsk.transform(X_train_tsk)
        X_val_tsk_scaled = scaler_tsk.transform(X_val_tsk)
        tsk_train_loader = build_loader(X_train_tsk_scaled, y_train, batch_size, device, task)
        tsk_val_loader = build_loader(X_val_tsk_scaled, y_val, batch_size, device, task)

        X_train_ga, ga_used = select_features_df(X_train_core, ga_selected, 'GA-ANFIS')
        X_val_ga = X_val_core.loc[:, ga_used] if ga_used is not None else X_val_core
        X_train_pso, pso_used = select_features_df(X_train_core, pso_selected, 'PSO-ANFIS')
        X_val_pso = X_val_core.loc[:, pso_used] if pso_used is not None else X_val_core

        scaler_ga = StandardScaler().fit(X_train_ga)
        X_train_ga_scaled = scaler_ga.transform(X_train_ga)
        X_val_ga_scaled = scaler_ga.transform(X_val_ga)
        ga_train_loader = build_loader(X_train_ga_scaled, y_train, batch_size, device, task)
        ga_val_loader = build_loader(X_val_ga_scaled, y_val, batch_size, device, task)

        scaler_pso = StandardScaler().fit(X_train_pso)
        X_train_pso_scaled = scaler_pso.transform(X_train_pso)
        X_val_pso_scaled = scaler_pso.transform(X_val_pso)
        pso_train_loader = build_loader(X_train_pso_scaled, y_train, batch_size, device, task)
        pso_val_loader = build_loader(X_val_pso_scaled, y_val, batch_size, device, task)

        feature_names_core = list(X_train_core.columns)

        gh_model, gh_criterion = fit_gh_anfis(
            gh_params, train_loader_core, X_train_core.shape[1], n_out, task, feature_names_core, device
        )
        tsk_model, _ = fit_tsk_anfis(
            tsk_params, tsk_train_loader, X_train_tsk.shape[1], n_out, task, device
        )
        ga_model, _ = fit_tsk_anfis(
            ga_params, ga_train_loader, X_train_ga.shape[1], n_out, task, device
        )
        pso_model, _ = fit_tsk_anfis(
            pso_params, pso_train_loader, X_train_pso.shape[1], n_out, task, device
        )

        parallel_models = {}
        parallel_meta = {}
        for ph_name, ph_fusion, ph_seed_offset in PH_VARIANTS:
            ph_split_seed = int(SEED + ph_seed_offset + int(fold_idx) * 997 + (0 if mi_top_k is None else int(mi_top_k)))
            ph_model, _, ph_info = fit_parallel_hier_anfis(
                ph_params,
                train_loader_core,
                X_train_core.shape[1],
                n_out,
                task,
                device,
                fusion=ph_fusion,
                split_seed=ph_split_seed,
                feature_names=feature_names_core,
                verbose=False,
            )
            parallel_models[ph_name] = ph_model
            parallel_meta[ph_name] = dict(ph_info or {})
            parallel_meta[ph_name]['split_seed'] = ph_split_seed

        svm_model = SVC(
            C=svm_params['C'],
            gamma=svm_params['gamma'],
            kernel=svm_params['kernel'],
            probability=True,
            random_state=SEED,
        )
        svm_model.fit(X_train_core, y_train)

        full_feature_names = list(X_train_core.columns) if hasattr(X_train_core, 'columns') else [f"x{i}" for i in range(X_train_core.shape[1])]
        tsk_feature_names = list(X_train_tsk.columns)
        ga_feature_names = list(X_train_ga.columns)
        pso_feature_names = list(X_train_pso.columns)

        dataset_artifact_name = f"{data_name}__{mode_label}"
        fold_artifact_paths = {
            'GH-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='GH-ANFIS',
                fold_idx=fold_idx,
                model=gh_model,
                params=gh_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
            ),
            'ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='ANFIS',
                fold_idx=fold_idx,
                model=tsk_model,
                params=tsk_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_tsk.shape[1],
                scaler=scaler_tsk,
                feature_names=tsk_feature_names,
            ),
            'GA-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='GA-ANFIS',
                fold_idx=fold_idx,
                model=ga_model,
                params=ga_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_ga.shape[1],
                scaler=scaler_ga,
                feature_names=ga_feature_names,
            ),
            'PSO-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PSO-ANFIS',
                fold_idx=fold_idx,
                model=pso_model,
                params=pso_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_pso.shape[1],
                scaler=scaler_pso,
                feature_names=pso_feature_names,
            ),
            'PH-ANFIS(Avg)': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PH-ANFIS(Avg)',
                fold_idx=fold_idx,
                model=parallel_models['PH-ANFIS(Avg)'],
                params={**ph_params, 'fusion': 'avg', 'split_seed': int((parallel_meta.get('PH-ANFIS(Avg)') or {}).get('split_seed', SEED + 101 + int(fold_idx) * 997))},
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
                extra_meta=parallel_meta.get('PH-ANFIS(Avg)'),
            ),
            'PH-ANFIS(Stacked)': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PH-ANFIS(Stacked)',
                fold_idx=fold_idx,
                model=parallel_models['PH-ANFIS(Stacked)'],
                params={**ph_params, 'fusion': 'stacked', 'split_seed': int((parallel_meta.get('PH-ANFIS(Stacked)') or {}).get('split_seed', SEED + 211 + int(fold_idx) * 997))},
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
                extra_meta=parallel_meta.get('PH-ANFIS(Stacked)'),
            ),
            'SVM': save_cv_sklearn_artifact(
                dataset_name=dataset_artifact_name,
                model_name='SVM',
                fold_idx=fold_idx,
                model=svm_model,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=None,
                feature_names=full_feature_names,
            ),
        }
        saved_cv_artifacts.append(fold_artifact_paths)
        print(f"Saved fold artifacts: {Path(next(iter(fold_artifact_paths.values()))).parent}")

        results = _eval_gh_detailed_cls(gh_model, val_loader_core, task)
        gh_base_metrics = results['base']
        gh_resid_metrics = results['residual']
        gh_full_metrics = results['combined']

        tsk_metrics = _eval_torch_cls(tsk_model, tsk_val_loader, task)
        ga_metrics = _eval_torch_cls(ga_model, ga_val_loader, task)
        pso_metrics = _eval_torch_cls(pso_model, pso_val_loader, task)
        ph_avg_metrics = _eval_torch_cls(parallel_models['PH-ANFIS(Avg)'], val_loader_core, task)
        ph_stk_metrics = _eval_torch_cls(parallel_models['PH-ANFIS(Stacked)'], val_loader_core, task)
        svm_metrics = _eval_sklearn_cls(svm_model, X_val_core, y_val, task)

        fold_metrics = {
            'GH-ANFIS': gh_full_metrics,
            'ANFIS': tsk_metrics,
            'GA-ANFIS': ga_metrics,
            'PSO-ANFIS': pso_metrics,
            'PH-ANFIS(Avg)': ph_avg_metrics,
            'PH-ANFIS(Stacked)': ph_stk_metrics,
            'SVM': svm_metrics,
        }

        print('Model results')
        for name in model_list:
            metrics = fold_metrics[name]
            print(
                f"{name:>18} | "
                f"Acc={metrics['acc']:.4f}, "
                f"Precision={metrics['precision']:.4f}, "
                f"Recall={metrics['recall']:.4f}, "
                f"F1={metrics['f1']:.4f}"
            )

        _push_metrics('GH-ANFIS(base)', gh_base_metrics)
        _push_metrics('GH-ANFIS(residual)', gh_resid_metrics)
        _push_metrics('GH-ANFIS(full)', gh_full_metrics)
        _push_metrics('ANFIS', tsk_metrics)
        _push_metrics('GA-ANFIS', ga_metrics)
        _push_metrics('PSO-ANFIS', pso_metrics)
        _push_metrics('PH-ANFIS(Avg)', ph_avg_metrics)
        _push_metrics('PH-ANFIS(Stacked)', ph_stk_metrics)
        _push_metrics('SVM', svm_metrics)

        gh_nauck = nauck_index_gh(gh_model, X_train_core_scaled)
        tsk_nauck = nauck_index_tsk(tsk_model, X_train_tsk_scaled)
        ga_nauck = nauck_index_tsk(ga_model, X_train_ga_scaled)
        pso_nauck = nauck_index_tsk(pso_model, X_train_pso_scaled)
        ph_avg_nauck = nauck_index_parallel_hier_tsk(parallel_models['PH-ANFIS(Avg)'], X_train_core_scaled)
        ph_stk_nauck = nauck_index_parallel_hier_tsk(parallel_models['PH-ANFIS(Stacked)'], X_train_core_scaled)

        base_info = gh_nauck.get('base') or {}
        resid_info = gh_nauck.get('residual') or {}
        overall_info = gh_nauck.get('overall') or {}

        base_rules = float(base_info.get('n_rules', 0) or 0)
        resid_rules = float(resid_info.get('n_rules', 0) or 0)
        total_rules = base_rules + resid_rules
        base_feats = float(base_info.get('n_features', 0) or 0)
        resid_feats = float(resid_info.get('n_features', 0) or 0)
        total_feats = base_feats + resid_feats

        def _weighted_by_count(b_val, b_count, r_val, r_count):
            total = b_count + r_count
            if total <= 0:
                return np.nan
            if np.isnan(b_val):
                b_val = 0.0
            if np.isnan(r_val):
                r_val = 0.0
            return (b_val * b_count + r_val * r_count) / total

        total_antecedents = base_rules * base_feats + resid_rules * resid_feats
        if total_rules > 0 and total_antecedents > 0:
            comp_overall = total_rules / total_antecedents
        else:
            comp_overall = np.nan

        cov_overall = _weighted_by_count(
            base_info.get('cov', np.nan), base_feats,
            resid_info.get('cov', np.nan), resid_feats,
        )
        part_overall = _weighted_by_count(
            base_info.get('part', np.nan), base_feats,
            resid_info.get('part', np.nan), resid_feats,
        )

        overall_info = {
            'index': overall_info.get('index', np.nan),
            'comp': comp_overall,
            'cov': cov_overall,
            'part': part_overall,
            'n_rules': total_rules if total_rules > 0 else overall_info.get('n_rules', np.nan),
            'n_features': total_feats if total_feats > 0 else overall_info.get('n_features', np.nan),
        }

        _push_nauck('GH-ANFIS(base)', base_info)
        _push_nauck('GH-ANFIS(residual)', resid_info)
        _push_nauck('GH-ANFIS(full)', overall_info)
        _push_nauck('ANFIS', tsk_nauck)
        _push_nauck('GA-ANFIS', ga_nauck)
        _push_nauck('PSO-ANFIS', pso_nauck)

        ph_avg_overall = dict((ph_avg_nauck.get('overall') or {}))
        ph_stk_overall = dict((ph_stk_nauck.get('overall') or {}))
        ph_avg_overall.setdefault('n_features', float(X_train_core.shape[1]))
        ph_stk_overall.setdefault('n_features', float(X_train_core.shape[1]))
        _push_nauck('PH-ANFIS(Avg)', ph_avg_overall)
        _push_nauck('PH-ANFIS(Stacked)', ph_stk_overall)

        _push_nauck('SVM', {
            'index': np.nan,
            'comp': np.nan,
            'cov': np.nan,
            'part': np.nan,
            'n_rules': np.nan,
            'n_features': float(X_train_core.shape[1]),
        })

        gh_overall = (gh_nauck.get('overall') or {}).get('index', np.nan)
        gh_base = (gh_nauck.get('base') or {}).get('index', np.nan)
        gh_resid = (gh_nauck.get('residual') or {}).get('index', np.nan)

        print('Nauck (overall)')
        print(f"GH-ANFIS | overall={gh_overall:.4f}, base={gh_base:.4f}, residual={gh_resid:.4f}")
        print(f"ANFIS    | overall={tsk_nauck.get('index', np.nan):.4f}")
        print(f"GA-ANFIS | overall={ga_nauck.get('index', np.nan):.4f}")
        print(f"PSO-ANFIS| overall={pso_nauck.get('index', np.nan):.4f}")
        print(f"PH-ANFIS(Avg)    | overall={(ph_avg_nauck.get('overall') or {}).get('index', np.nan):.4f}")
        print(f"PH-ANFIS(Stacked)| overall={(ph_stk_nauck.get('overall') or {}).get('index', np.nan):.4f}")

    print(f"CV weight root: {CV_WEIGHT_ROOT.resolve()}")
    print(f"Saved fold artifacts: {len(saved_cv_artifacts)} folds")
    print(f"CV Summary (mean+-std) | {data_name} | {mode_label}")

    def _mean_std(values):
        arr = np.asarray(values, dtype=float)
        if arr.size == 0:
            return float('nan'), float('nan')
        return float(np.nanmean(arr)), float(np.nanstd(arr))

    rows = []
    for name in summary_models:
        acc_mean, acc_std = _mean_std(cv_results[name]['acc'])
        precision_mean, precision_std = _mean_std(cv_results[name]['precision'])
        recall_mean, recall_std = _mean_std(cv_results[name]['recall'])
        f1_mean, f1_std = _mean_std(cv_results[name]['f1'])
        idx_mean, idx_std = _mean_std(cv_nauck[name]['index'])
        comp_mean, comp_std = _mean_std(cv_nauck[name]['comp'])
        cov_mean, cov_std = _mean_std(cv_nauck[name]['cov'])
        part_mean, part_std = _mean_std(cv_nauck[name]['part'])
        rules_mean, rules_std = _mean_std(cv_nauck[name]['n_rules'])
        feats_mean, feats_std = _mean_std(cv_nauck[name]['n_features'])

        print(
            f"{name:>18} | "
            f"Acc={acc_mean:.4f}+-{acc_std:.4f}, "
            f"Precision={precision_mean:.4f}+-{precision_std:.4f}, "
            f"Recall={recall_mean:.4f}+-{recall_std:.4f}, "
            f"F1={f1_mean:.4f}+-{f1_std:.4f} | "
            f"Nauck idx={idx_mean:.4f}+-{idx_std:.4f}, "
            f"comp={comp_mean:.4f}+-{comp_std:.4f}, "
            f"cov={cov_mean:.4f}+-{cov_std:.4f}, "
            f"part={part_mean:.4f}+-{part_std:.4f}, "
            f"n_rules={rules_mean:.4f}+-{rules_std:.4f}, "
            f"allowed_features={feats_mean:.4f}+-{feats_std:.4f}"
        )

        rows.append({
            'dataset': data_name,
            'mode': mode_label,
            'model': name,
            'acc_mean': acc_mean,
            'acc_std': acc_std,
            'precision_mean': precision_mean,
            'precision_std': precision_std,
            'recall_mean': recall_mean,
            'recall_std': recall_std,
            'f1_mean': f1_mean,
            'f1_std': f1_std,
            'nauck_mean': idx_mean,
            'nauck_std': idx_std,
            'comp_mean': comp_mean,
            'comp_std': comp_std,
            'cov_mean': cov_mean,
            'cov_std': cov_std,
            'part_mean': part_mean,
            'part_std': part_std,
            'n_rules_mean': rules_mean,
            'n_rules_std': rules_std,
            'n_features_mean': feats_mean,
            'n_features_std': feats_std,
        })

    return pd.DataFrame(rows)


In [ ]:
GH_PARAMS_BY_DATASET = {
    'Vowel': {'lr_base': 0.1,
        'lr_residual': 0.01,
        'residual_rules': 11,
        'base_rules': 6,
        'mf_per_feature': 2,
        'epochs_stage1': 130,
        'epochs_stage2': 50,
        'lambda_resid_s2': 0.01,
        'lambda_base_s1': 0.01,
        'weight_decay': 1e-05,
        'base_hard_epochs': 80,
        'residual_hard_epochs': 30,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'rule_init_mode': 'balanced',
        'rule_seed': 0,
        'firing_mode': 'htsk',
        'residual_gate_mode': 'complement',
        'use_input_norm': False,
        'enable_residual_branch': True,
        'random_role_assignment': False},
    'Breast_Cancer_Wisconsin_(Original)': {'lr_base': 0.1,
        'lr_residual': 0.005,
        'residual_rules': 11,
        'base_rules': 7,
        'mf_per_feature': 2,
        'epochs_stage1': 50,
        'epochs_stage2': 30,
        'lambda_resid_s2': 0.0,
        'lambda_base_s1': 0.01,
        'weight_decay': 1e-05,
        'base_hard_epochs': 10,
        'residual_hard_epochs': 10,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'rule_init_mode': 'balanced',
        'rule_seed': 0,
        'firing_mode': 'htsk',
        'residual_gate_mode': 'complement',
        'use_input_norm': False,
        'enable_residual_branch': True,
        'random_role_assignment': False},
    'Spambase': {'lr_base': 0.1,
        'lr_residual': 0.01,
        'residual_rules': 11,
        'base_rules': 7,
        'mf_per_feature': 3,
        'epochs_stage1': 50,
        'epochs_stage2': 50,
        'lambda_resid_s2': 0.001,
        'lambda_base_s1': 0.001,
        'weight_decay': 1e-06,
        'base_hard_epochs': 50,
        'residual_hard_epochs': 50,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'firing_mode':'htsk'},
    'Gisette': {
        'lr_base': 0.01,
        'lr_residual': 0.01,
        'residual_rules': 11,
        'base_rules': 7,
        'mf_per_feature': 2,
        'epochs_stage1': 30,
        'epochs_stage2': 30,
        'lambda_resid_s2': 0.001,
        'lambda_base_s1': 0.01,
        'weight_decay': 0.0001,
        'base_hard_epochs': 20,
        'residual_hard_epochs': 10,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'firing_mode':'htsk'
    },
}


In [ ]:
NO_MI_RESULTS = []
# dataset_list = ['Breast_Cancer_Wisconsin_(Original)']
for data_name in dataset_list:
    gh_params = copy.deepcopy(GH_PARAMS_BY_DATASET[data_name])
    df_mode = run_cv_dataset_mode(
        data_name=data_name,
        gh_params=gh_params,
        mi_top_k=None,
        mode_label='no_mi',
    )
    NO_MI_RESULTS.append(df_mode)

NO_MI_SUMMARY = pd.concat(NO_MI_RESULTS, ignore_index=True)
print(f"[no_mi] rows: {len(NO_MI_SUMMARY)}")
NO_MI_SUMMARY


In [ ]:
# MI100_RESULTS = []

# for data_name in dataset_list:
#     gh_params = copy.deepcopy(GH_PARAMS_BY_DATASET[data_name])
#     df_mode = run_cv_dataset_mode(
#         data_name=data_name,
#         gh_params=gh_params,
#         mi_top_k=100,
#         mode_label='mi_top100',
#     )
#     MI100_RESULTS.append(df_mode)

# MI100_SUMMARY = pd.concat(MI100_RESULTS, ignore_index=True)
# print(f"[mi_top100] rows: {len(MI100_SUMMARY)}")
# MI100_SUMMARY


In [ ]:
# ALL_SUMMARY = pd.concat([NO_MI_SUMMARY, MI100_SUMMARY], ignore_index=True)
ALL_SUMMARY = pd.concat([NO_MI_SUMMARY], ignore_index=True)
out_dir = Path('./outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'maincode_with_ph_no_mi_summary.csv'
ALL_SUMMARY.to_csv(out_path, index=False)

print(f"Saved: {out_path.resolve()}")
print(f"Total rows: {len(ALL_SUMMARY)}")
ALL_SUMMARY


In [ ]:
best_acc = (
    ALL_SUMMARY[ALL_SUMMARY["model"] != "PH-ANFIS(Stacked)"]
    .loc[
        lambda df: df.groupby(["dataset", "mode"])["acc_mean"].idxmax(),
        ["dataset", "mode", "model", "acc_mean"],
    ]
)
best_acc

## BCWD 5-fold performance workflow


# BCWD 5-Fold Experiment

BCWD만 대상으로 5-fold 교차검증을 수행하는 전용 노트북입니다.

- 입력은 `load_bcwd_data()`를 그대로 사용하므로, E404 전처리 기준에서 BCWD는 one-hot 확장 없이 `9`개 numeric feature로 학습됩니다.
- 실행 대상 모델은 `GH-ANFIS`, `ANFIS`, `GA-ANFIS`, `PSO-ANFIS`, `PH-ANFIS(Avg)`, `PH-ANFIS(Stacked)`, `SVM` 입니다.
- fold별 가중치 아티팩트는 `hyper_parameter/cv_weights/Breast_Cancer_Wisconsin__Original___<mode>/fold_XX/` 아래에 저장됩니다.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()

def _looks_like_project_root(path):
    markers = ["model.py", "data.py", "learning.py", "utils.py"]
    return all((path / marker).exists() for marker in markers)

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E404",
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E404",
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)

os.environ.setdefault("OPENML_DATA_HOME", str(PROJECT_ROOT / "data" / "openml_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
(PROJECT_ROOT / "data" / "openml_cache").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS, TSKANFIS, ParallelHierarchicalTSKANFIS
from utils import set_deterministic, build_loader, load_gh_params, load_ga_params, load_pso_params
from data import *
from learning import *
from gh_eval_utils import *
from interpretability import nauck_index_gh, nauck_index_tsk, nauck_index_parallel_hier_tsk
from sklearn.svm import SVC
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from tqdm import tqdm

print(f"Project root: {PROJECT_ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

set_deterministic(SEED)


In [ ]:
def _metric_average(task_kind):
    return 'weighted' if task_kind == 'multiclass' else 'binary'


def _compute_cls_metrics(y_true, y_pred, task_kind):
    average = _metric_average(task_kind)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
        'recall': recall_score(y_true, y_pred, average=average, zero_division=0),
        'f1': f1_score(y_true, y_pred, average=average, zero_division=0),
    }


def _eval_torch_cls(model, loader, task_kind):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device).float()
            y_batch = y_batch.to(device)
            logits = model(x_batch)

            if task_kind == 'binary':
                if logits.dim() == 1:
                    logits = logits.unsqueeze(1)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long().squeeze(1)
                targets = y_batch.long().squeeze(1) if y_batch.dim() > 1 else y_batch.long()
            else:
                preds = torch.argmax(logits, dim=1)
                targets = y_batch.long()

            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_targets, axis=0)
    return _compute_cls_metrics(y_true, y_pred, task_kind)


def _eval_sklearn_cls(model, X, y, task_kind):
    preds = model.predict(X)
    return _compute_cls_metrics(y, preds, task_kind)


def _eval_gh_detailed_cls(model, loader, task_kind):
    results = {}

    for mode_name in ['base_only', 'residual_only', 'full']:
        model.set_mode(mode_name)
        model.eval()

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for x_batch, y_batch in loader:
                x_batch = x_batch.to(device).float()
                y_batch = y_batch.to(device)
                logits = model(x_batch)

                if task_kind == 'binary':
                    if logits.dim() == 1:
                        logits = logits.unsqueeze(1)
                    probs = torch.sigmoid(logits)
                    preds = (probs >= 0.5).long().squeeze(1)
                    targets = y_batch.long().squeeze(1) if y_batch.dim() > 1 else y_batch.long()
                else:
                    preds = torch.argmax(logits, dim=1)
                    targets = y_batch.long()

                all_preds.append(preds.cpu().numpy())
                all_targets.append(targets.cpu().numpy())

        y_pred = np.concatenate(all_preds, axis=0)
        y_true = np.concatenate(all_targets, axis=0)
        key = 'combined' if mode_name == 'full' else mode_name.replace('_only', '')
        results[key] = _compute_cls_metrics(y_true, y_pred, task_kind)

    model.set_mode('full')
    return results



from pathlib import Path
import copy
import joblib

CV_WEIGHT_ROOT = Path("./hyper_parameter/cv_weights")
_MODEL_FILE_STEM = {
    "GH-ANFIS": "gh_anfis",
    "ANFIS": "anfis",
    "GA-ANFIS": "ga_anfis",
    "PSO-ANFIS": "pso_anfis",
    "PH-ANFIS(Avg)": "ph_anfis_avg",
    "PH-ANFIS(Stacked)": "ph_anfis_stacked",
    "SVM": "svm",
}


def _safe_name(name):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(name))


def _scaler_stats(scaler):
    if scaler is None:
        return None, None
    mean = getattr(scaler, "mean_", None)
    scale = getattr(scaler, "scale_", None)
    return (mean.tolist() if mean is not None else None, scale.tolist() if scale is not None else None)


def save_cv_torch_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    params,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    scaler_mean, scaler_scale = _scaler_stats(scaler)
    stem = _MODEL_FILE_STEM[model_name]
    out_path = fold_dir / f"{stem}.pt"

    state_dict_cpu = {
        k: (v.detach().cpu() if torch.is_tensor(v) else v)
        for k, v in model.state_dict().items()
    }

    payload = {
        "framework": "torch",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "params": copy.deepcopy(params),
        "scaler_mean": scaler_mean,
        "scaler_scale": scaler_scale,
        "state_dict": state_dict_cpu,
        "extra_meta": dict(extra_meta or {}),
    }
    torch.save(payload, out_path)
    return str(out_path)


def save_cv_sklearn_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    stem = _MODEL_FILE_STEM[model_name]
    out_path = fold_dir / f"{stem}.joblib"

    payload = {
        "framework": "sklearn",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "scaler": scaler,
        "model": model,
        "extra_meta": dict(extra_meta or {}),
    }
    joblib.dump(payload, out_path)
    return str(out_path)


def load_cv_artifact(dataset_name, model_name, fold_idx, root_dir=CV_WEIGHT_ROOT, device=device):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    stem = _MODEL_FILE_STEM[model_name]

    if model_name == "SVM":
        payload = joblib.load(fold_dir / f"{stem}.joblib")
        model = payload["model"]
        return model, payload

    payload = torch.load(fold_dir / f"{stem}.pt", map_location=device)
    params = payload.get("params") or {}
    n_features = int(payload["n_features"])
    n_outputs = int(payload["n_outputs"])

    if model_name == "GH-ANFIS":
        model = GH_ANFIS(
            n_features=n_features,
            n_outputs=n_outputs,
            residual_rules=int(params.get("residual_rules", 8)),
            base_rules=int(params.get("base_rules", 4)),
            mf_per_feature=int(params.get("mf_per_feature", 2)),
            device=device,
        ).to(device)
    elif model_name in ("PH-ANFIS(Avg)", "PH-ANFIS(Stacked)"):
        fusion_default = "avg" if model_name == "PH-ANFIS(Avg)" else "stacked"
        fusion = str(params.get("fusion", fusion_default))
        extra_meta = payload.get("extra_meta") or {}
        group_a_idx = list(extra_meta.get("group_a_idx") or [])
        group_b_idx = list(extra_meta.get("group_b_idx") or [])
        if not group_a_idx or not group_b_idx:
            split_seed = int(params.get("split_seed", SEED))
            rng = np.random.default_rng(split_seed)
            order = np.arange(int(n_features), dtype=int)
            rng.shuffle(order)
            cut = int(max(1, n_features // 2))
            if cut >= n_features:
                cut = n_features - 1
            group_a_idx = np.sort(order[:cut]).tolist()
            group_b_idx = np.sort(order[cut:]).tolist()

        model = ParallelHierarchicalTSKANFIS(
            n_inputs=n_features,
            n_outputs=n_outputs,
            group_a_idx=group_a_idx,
            group_b_idx=group_b_idx,
            branch_rules=int(params.get("branch_rules", params.get("n_rules", 12))),
            top_rules=int(params.get("top_rules", max(2, int(params.get("branch_rules", params.get("n_rules", 12))) // 2))),
            fusion=fusion,
            mfs_per_input=int(params.get("mfs_per_input", 3)),
        ).to(device)
    else:
        model = TSKANFIS(
            n_inputs=n_features,
            n_rules=int(params.get("n_rules", 30)),
            n_outputs=n_outputs,
            mfs_per_input=int(params.get("mfs_per_input", 3)),
        ).to(device)

    model.load_state_dict(payload["state_dict"])
    model.eval()
    return model, payload


def build_scaler_from_artifact(payload):
    mean = payload.get("scaler_mean")
    scale = payload.get("scaler_scale")
    if mean is None or scale is None:
        return None
    scaler = StandardScaler()
    scaler.mean_ = np.asarray(mean, dtype=float)
    scaler.scale_ = np.asarray(scale, dtype=float)
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = int(scaler.mean_.shape[0])
    scaler.n_samples_seen_ = 1
    return scaler


def _normalize_feature_key(name):
    text = str(name).strip()
    return "_".join(part for part in ''.join(ch if ch.isalnum() else '_' for ch in text).split('_') if part).lower()


def select_features_df(X, selected, label):
    if not selected:
        return X, list(X.columns) if hasattr(X, "columns") else None
    if not hasattr(X, "columns"):
        return X, None

    available_cols = list(X.columns)
    exact_map = {str(col): col for col in available_cols}
    normalized_map = {}
    for col in available_cols:
        normalized_map.setdefault(_normalize_feature_key(col), []).append(col)

    resolved = []
    missing = []

    for feat in selected:
        feat_str = str(feat)
        if feat_str in exact_map:
            resolved.append(exact_map[feat_str])
            continue

        if feat_str.startswith("x") and feat_str[1:].isdigit():
            idx = int(feat_str[1:])
            if idx < X.shape[1]:
                resolved.append(X.columns[idx])
                continue

        candidates = normalized_map.get(_normalize_feature_key(feat_str), [])
        if len(candidates) == 1:
            resolved.append(candidates[0])
            continue

        missing.append(feat)

    resolved = list(dict.fromkeys(resolved))
    if resolved:
        if missing:
            print(f"[{label}] matched {len(resolved)} selected_features; skipped {len(missing)} unmatched entries")
        return X.loc[:, resolved], list(resolved)

    print(f"[{label}] selected_features not found; using all features")
    return X, list(X.columns)


PH_PARAMS_BY_DATASET = {
    "Vowel": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 18,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Breast_Cancer_Wisconsin_(Original)": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 16,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Spambase": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 20,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Gisette": {
        "lr": 0.03,
        "epochs": 70,
        "branch_rules": 14,
        "top_rules": 6,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
}

PH_VARIANTS = [
    ("PH-ANFIS(Avg)", "avg", 101),
    ("PH-ANFIS(Stacked)", "stacked", 211),
]


In [ ]:
DATASET_NAME = 'Breast_Cancer_Wisconsin_(Original)'
MODE_LABEL = 'no_mi'
MI_TOP_K = None
MAX_FOLDS = None  # smoke test가 필요하면 1처럼 줄여서 실행
BATCH_SIZE = 1024
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'bcwd_5fold_experiment'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_list = [
    'GH-ANFIS',
    'ANFIS',
    'GA-ANFIS',
    'PSO-ANFIS',
    'PH-ANFIS(Avg)',
    'PH-ANFIS(Stacked)',
    'SVM',
]

BCWD_x, BCWD_y, BCWD_feature_name = load_bcwd_data()
BCWD_x = coerce_numeric_frame(BCWD_x)
BCWD_x, BCWD_y = drop_nan_targets(BCWD_x, BCWD_y)
BCWD_x = BCWD_x.copy()
BCWD_y = np.asarray(BCWD_y, dtype=np.int64)
BCWD_feature_name = list(BCWD_x.columns)

print('BCWD shape:', BCWD_x.shape)
print('BCWD features:', BCWD_feature_name)
print('Class counts:')
print(pd.Series(BCWD_y).value_counts().sort_index())


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd


def _top_mi_columns(X_train_df, y_train, top_k, random_state=SEED):
    if top_k is None:
        return list(X_train_df.columns)
    k = int(max(1, min(int(top_k), X_train_df.shape[1])))
    if k >= X_train_df.shape[1]:
        return list(X_train_df.columns)

    mi = mutual_info_classif(
        X_train_df,
        np.asarray(y_train).astype(int),
        random_state=int(random_state),
    )
    order = np.argsort(mi)[::-1]
    return [X_train_df.columns[i] for i in order[:k]]


def _dataset_bundle(data_name):
    if data_name != DATASET_NAME:
        raise ValueError(f'This notebook is BCWD-only. Received: {data_name}')
    return BCWD_x, BCWD_y, BCWD_feature_name


def _batch_size_for_dataset(data_name):
    return BATCH_SIZE


def run_bcwd_cv_dataset_mode(data_name, gh_params, mi_top_k=None, mode_label='no_mi', max_folds=None):
    X_df, y, feature_name = _dataset_bundle(data_name)

    n_classes = len(np.unique(y))
    task = 'binary' if n_classes == 2 else 'multiclass'
    n_out = 1 if task == 'binary' else n_classes
    n_folds = 5
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    ga_params = load_ga_params(data_name, data_name)
    pso_params = load_pso_params(data_name, data_name)
    tsk_params = {'lr': 0.1, 'n_rules': 30, 'epochs': 100, 'selected_features': []}
    ph_params = copy.deepcopy(PH_PARAMS_BY_DATASET[data_name])
    svm_params = {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}

    ga_selected = list(ga_params.get('selected_features') or [])
    pso_selected = list(pso_params.get('selected_features') or [])

    if data_name == 'Gisette' and mi_top_k is not None and mi_top_k > 0:
        ga_selected = ga_selected[:int(mi_top_k)]
        pso_selected = pso_selected[:int(mi_top_k)]

    print(f"[{data_name}/{mode_label}] ANFIS selected=core, GA selected={len(ga_selected)}, PSO selected={len(pso_selected)}")

    summary_models = [
        'GH-ANFIS(base)',
        'GH-ANFIS(residual)',
        'GH-ANFIS(full)',
        'ANFIS',
        'GA-ANFIS',
        'PSO-ANFIS',
        'PH-ANFIS(Avg)',
        'PH-ANFIS(Stacked)',
        'SVM',
    ]

    metric_names = ('acc', 'precision', 'recall', 'f1')
    cv_results = {name: {metric: [] for metric in metric_names} for name in summary_models}
    cv_nauck = {
        name: {'index': [], 'comp': [], 'cov': [], 'part': [], 'n_rules': [], 'n_features': []}
        for name in summary_models
    }

    def _push_nauck(name, info):
        cv_nauck[name]['index'].append(info.get('index', np.nan))
        cv_nauck[name]['comp'].append(info.get('comp', np.nan))
        cv_nauck[name]['cov'].append(info.get('cov', np.nan))
        cv_nauck[name]['part'].append(info.get('part', np.nan))
        cv_nauck[name]['n_rules'].append(info.get('n_rules', np.nan))
        cv_nauck[name]['n_features'].append(info.get('n_features', np.nan))

    def _push_metrics(name, metrics):
        for metric in metric_names:
            cv_results[name][metric].append(metrics.get(metric, np.nan))

    saved_cv_artifacts = []

    X_train_df, X_test_df, y_train_tmp, y_test_tmp = train_test_split(
        X_df, y, test_size=0.2, random_state=SEED
    )
    scaler_tmp = StandardScaler().fit(X_train_df)
    X_train_tmp = scaler_tmp.transform(X_train_df)
    X_test_tmp = scaler_tmp.transform(X_test_df)
    print(f"[{data_name}/{mode_label}] Train: {X_train_tmp.shape}, Test: {X_test_tmp.shape}")

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_df, y), 1):
        if max_folds is not None and int(fold_idx) > int(max_folds):
            break
        set_deterministic(SEED)
        print(f"{'='*60}")
        print(f"[{data_name}/{mode_label}] Fold {fold_idx}/{n_folds}")
        print(f"{'='*60}")

        X_train_full, X_val_full = X_df.iloc[train_idx], X_df.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if mi_top_k is None:
            X_train_core = X_train_full
            X_val_core = X_val_full
        else:
            mi_cols = _top_mi_columns(X_train_full, y_train, mi_top_k, random_state=SEED)
            X_train_core = X_train_full.loc[:, mi_cols]
            X_val_core = X_val_full.loc[:, mi_cols]
            if fold_idx == 1:
                print(f"[{data_name}/{mode_label}] MI selected={len(mi_cols)}")

        batch_size = _batch_size_for_dataset(data_name)

        scaler_core = StandardScaler().fit(X_train_core)
        X_train_core_scaled = scaler_core.transform(X_train_core)
        X_val_core_scaled = scaler_core.transform(X_val_core)
        train_loader_core = build_loader(X_train_core_scaled, y_train, batch_size, device, task)
        val_loader_core = build_loader(X_val_core_scaled, y_val, batch_size, device, task)

        X_train_tsk = X_train_core
        X_val_tsk = X_val_core
        tsk_used = list(X_train_core.columns)
        scaler_tsk = StandardScaler().fit(X_train_tsk)
        X_train_tsk_scaled = scaler_tsk.transform(X_train_tsk)
        X_val_tsk_scaled = scaler_tsk.transform(X_val_tsk)
        tsk_train_loader = build_loader(X_train_tsk_scaled, y_train, batch_size, device, task)
        tsk_val_loader = build_loader(X_val_tsk_scaled, y_val, batch_size, device, task)

        X_train_ga, ga_used = select_features_df(X_train_core, ga_selected, 'GA-ANFIS')
        X_val_ga = X_val_core.loc[:, ga_used] if ga_used is not None else X_val_core
        X_train_pso, pso_used = select_features_df(X_train_core, pso_selected, 'PSO-ANFIS')
        X_val_pso = X_val_core.loc[:, pso_used] if pso_used is not None else X_val_core

        scaler_ga = StandardScaler().fit(X_train_ga)
        X_train_ga_scaled = scaler_ga.transform(X_train_ga)
        X_val_ga_scaled = scaler_ga.transform(X_val_ga)
        ga_train_loader = build_loader(X_train_ga_scaled, y_train, batch_size, device, task)
        ga_val_loader = build_loader(X_val_ga_scaled, y_val, batch_size, device, task)

        scaler_pso = StandardScaler().fit(X_train_pso)
        X_train_pso_scaled = scaler_pso.transform(X_train_pso)
        X_val_pso_scaled = scaler_pso.transform(X_val_pso)
        pso_train_loader = build_loader(X_train_pso_scaled, y_train, batch_size, device, task)
        pso_val_loader = build_loader(X_val_pso_scaled, y_val, batch_size, device, task)

        feature_names_core = list(X_train_core.columns)

        gh_model, gh_criterion = fit_gh_anfis(
            gh_params, train_loader_core, X_train_core.shape[1], n_out, task, feature_names_core, device
        )
        tsk_model, _ = fit_tsk_anfis(
            tsk_params, tsk_train_loader, X_train_tsk.shape[1], n_out, task, device
        )
        ga_model, _ = fit_tsk_anfis(
            ga_params, ga_train_loader, X_train_ga.shape[1], n_out, task, device
        )
        pso_model, _ = fit_tsk_anfis(
            pso_params, pso_train_loader, X_train_pso.shape[1], n_out, task, device
        )

        parallel_models = {}
        parallel_meta = {}
        for ph_name, ph_fusion, ph_seed_offset in PH_VARIANTS:
            ph_split_seed = int(SEED + ph_seed_offset + int(fold_idx) * 997 + (0 if mi_top_k is None else int(mi_top_k)))
            ph_model, _, ph_info = fit_parallel_hier_anfis(
                ph_params,
                train_loader_core,
                X_train_core.shape[1],
                n_out,
                task,
                device,
                fusion=ph_fusion,
                split_seed=ph_split_seed,
                feature_names=feature_names_core,
                verbose=False,
            )
            parallel_models[ph_name] = ph_model
            parallel_meta[ph_name] = dict(ph_info or {})
            parallel_meta[ph_name]['split_seed'] = ph_split_seed

        svm_model = SVC(
            C=svm_params['C'],
            gamma=svm_params['gamma'],
            kernel=svm_params['kernel'],
            probability=True,
            random_state=SEED,
        )
        svm_model.fit(X_train_core, y_train)

        full_feature_names = list(X_train_core.columns) if hasattr(X_train_core, 'columns') else [f"x{i}" for i in range(X_train_core.shape[1])]
        tsk_feature_names = list(X_train_tsk.columns)
        ga_feature_names = list(X_train_ga.columns)
        pso_feature_names = list(X_train_pso.columns)

        dataset_artifact_name = f"{data_name}__{mode_label}"
        fold_artifact_paths = {
            'GH-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='GH-ANFIS',
                fold_idx=fold_idx,
                model=gh_model,
                params=gh_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
            ),
            'ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='ANFIS',
                fold_idx=fold_idx,
                model=tsk_model,
                params=tsk_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_tsk.shape[1],
                scaler=scaler_tsk,
                feature_names=tsk_feature_names,
            ),
            'GA-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='GA-ANFIS',
                fold_idx=fold_idx,
                model=ga_model,
                params=ga_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_ga.shape[1],
                scaler=scaler_ga,
                feature_names=ga_feature_names,
            ),
            'PSO-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PSO-ANFIS',
                fold_idx=fold_idx,
                model=pso_model,
                params=pso_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_pso.shape[1],
                scaler=scaler_pso,
                feature_names=pso_feature_names,
            ),
            'PH-ANFIS(Avg)': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PH-ANFIS(Avg)',
                fold_idx=fold_idx,
                model=parallel_models['PH-ANFIS(Avg)'],
                params={**ph_params, 'fusion': 'avg', 'split_seed': int((parallel_meta.get('PH-ANFIS(Avg)') or {}).get('split_seed', SEED + 101 + int(fold_idx) * 997))},
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
                extra_meta=parallel_meta.get('PH-ANFIS(Avg)'),
            ),
            'PH-ANFIS(Stacked)': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PH-ANFIS(Stacked)',
                fold_idx=fold_idx,
                model=parallel_models['PH-ANFIS(Stacked)'],
                params={**ph_params, 'fusion': 'stacked', 'split_seed': int((parallel_meta.get('PH-ANFIS(Stacked)') or {}).get('split_seed', SEED + 211 + int(fold_idx) * 997))},
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
                extra_meta=parallel_meta.get('PH-ANFIS(Stacked)'),
            ),
            'SVM': save_cv_sklearn_artifact(
                dataset_name=dataset_artifact_name,
                model_name='SVM',
                fold_idx=fold_idx,
                model=svm_model,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=None,
                feature_names=full_feature_names,
            ),
        }
        saved_cv_artifacts.append(fold_artifact_paths)
        print(f"Saved fold artifacts: {Path(next(iter(fold_artifact_paths.values()))).parent}")

        results = _eval_gh_detailed_cls(gh_model, val_loader_core, task)
        gh_base_metrics = results['base']
        gh_resid_metrics = results['residual']
        gh_full_metrics = results['combined']

        tsk_metrics = _eval_torch_cls(tsk_model, tsk_val_loader, task)
        ga_metrics = _eval_torch_cls(ga_model, ga_val_loader, task)
        pso_metrics = _eval_torch_cls(pso_model, pso_val_loader, task)
        ph_avg_metrics = _eval_torch_cls(parallel_models['PH-ANFIS(Avg)'], val_loader_core, task)
        ph_stk_metrics = _eval_torch_cls(parallel_models['PH-ANFIS(Stacked)'], val_loader_core, task)
        svm_metrics = _eval_sklearn_cls(svm_model, X_val_core, y_val, task)

        fold_metrics = {
            'GH-ANFIS': gh_full_metrics,
            'ANFIS': tsk_metrics,
            'GA-ANFIS': ga_metrics,
            'PSO-ANFIS': pso_metrics,
            'PH-ANFIS(Avg)': ph_avg_metrics,
            'PH-ANFIS(Stacked)': ph_stk_metrics,
            'SVM': svm_metrics,
        }

        print('Model results')
        for name in model_list:
            metrics = fold_metrics[name]
            print(
                f"{name:>18} | "
                f"Acc={metrics['acc']:.4f}, "
                f"Precision={metrics['precision']:.4f}, "
                f"Recall={metrics['recall']:.4f}, "
                f"F1={metrics['f1']:.4f}"
            )

        _push_metrics('GH-ANFIS(base)', gh_base_metrics)
        _push_metrics('GH-ANFIS(residual)', gh_resid_metrics)
        _push_metrics('GH-ANFIS(full)', gh_full_metrics)
        _push_metrics('ANFIS', tsk_metrics)
        _push_metrics('GA-ANFIS', ga_metrics)
        _push_metrics('PSO-ANFIS', pso_metrics)
        _push_metrics('PH-ANFIS(Avg)', ph_avg_metrics)
        _push_metrics('PH-ANFIS(Stacked)', ph_stk_metrics)
        _push_metrics('SVM', svm_metrics)

        gh_nauck = nauck_index_gh(gh_model, X_train_core_scaled)
        tsk_nauck = nauck_index_tsk(tsk_model, X_train_tsk_scaled)
        ga_nauck = nauck_index_tsk(ga_model, X_train_ga_scaled)
        pso_nauck = nauck_index_tsk(pso_model, X_train_pso_scaled)
        ph_avg_nauck = nauck_index_parallel_hier_tsk(parallel_models['PH-ANFIS(Avg)'], X_train_core_scaled)
        ph_stk_nauck = nauck_index_parallel_hier_tsk(parallel_models['PH-ANFIS(Stacked)'], X_train_core_scaled)

        base_info = gh_nauck.get('base') or {}
        resid_info = gh_nauck.get('residual') or {}
        overall_info = gh_nauck.get('overall') or {}

        base_rules = float(base_info.get('n_rules', 0) or 0)
        resid_rules = float(resid_info.get('n_rules', 0) or 0)
        total_rules = base_rules + resid_rules
        base_feats = float(base_info.get('n_features', 0) or 0)
        resid_feats = float(resid_info.get('n_features', 0) or 0)
        total_feats = base_feats + resid_feats

        def _weighted_by_count(b_val, b_count, r_val, r_count):
            total = b_count + r_count
            if total <= 0:
                return np.nan
            if np.isnan(b_val):
                b_val = 0.0
            if np.isnan(r_val):
                r_val = 0.0
            return (b_val * b_count + r_val * r_count) / total

        total_antecedents = base_rules * base_feats + resid_rules * resid_feats
        if total_rules > 0 and total_antecedents > 0:
            comp_overall = total_rules / total_antecedents
        else:
            comp_overall = np.nan

        cov_overall = _weighted_by_count(
            base_info.get('cov', np.nan), base_feats,
            resid_info.get('cov', np.nan), resid_feats,
        )
        part_overall = _weighted_by_count(
            base_info.get('part', np.nan), base_feats,
            resid_info.get('part', np.nan), resid_feats,
        )

        overall_info = {
            'index': overall_info.get('index', np.nan),
            'comp': comp_overall,
            'cov': cov_overall,
            'part': part_overall,
            'n_rules': total_rules if total_rules > 0 else overall_info.get('n_rules', np.nan),
            'n_features': total_feats if total_feats > 0 else overall_info.get('n_features', np.nan),
        }

        _push_nauck('GH-ANFIS(base)', base_info)
        _push_nauck('GH-ANFIS(residual)', resid_info)
        _push_nauck('GH-ANFIS(full)', overall_info)
        _push_nauck('ANFIS', tsk_nauck)
        _push_nauck('GA-ANFIS', ga_nauck)
        _push_nauck('PSO-ANFIS', pso_nauck)

        ph_avg_overall = dict((ph_avg_nauck.get('overall') or {}))
        ph_stk_overall = dict((ph_stk_nauck.get('overall') or {}))
        ph_avg_overall.setdefault('n_features', float(X_train_core.shape[1]))
        ph_stk_overall.setdefault('n_features', float(X_train_core.shape[1]))
        _push_nauck('PH-ANFIS(Avg)', ph_avg_overall)
        _push_nauck('PH-ANFIS(Stacked)', ph_stk_overall)

        _push_nauck('SVM', {
            'index': np.nan,
            'comp': np.nan,
            'cov': np.nan,
            'part': np.nan,
            'n_rules': np.nan,
            'n_features': float(X_train_core.shape[1]),
        })

        gh_overall = (gh_nauck.get('overall') or {}).get('index', np.nan)
        gh_base = (gh_nauck.get('base') or {}).get('index', np.nan)
        gh_resid = (gh_nauck.get('residual') or {}).get('index', np.nan)

        print('Nauck (overall)')
        print(f"GH-ANFIS | overall={gh_overall:.4f}, base={gh_base:.4f}, residual={gh_resid:.4f}")
        print(f"ANFIS    | overall={tsk_nauck.get('index', np.nan):.4f}")
        print(f"GA-ANFIS | overall={ga_nauck.get('index', np.nan):.4f}")
        print(f"PSO-ANFIS| overall={pso_nauck.get('index', np.nan):.4f}")
        print(f"PH-ANFIS(Avg)    | overall={(ph_avg_nauck.get('overall') or {}).get('index', np.nan):.4f}")
        print(f"PH-ANFIS(Stacked)| overall={(ph_stk_nauck.get('overall') or {}).get('index', np.nan):.4f}")

    print(f"CV weight root: {CV_WEIGHT_ROOT.resolve()}")
    print(f"Saved fold artifacts: {len(saved_cv_artifacts)} folds")
    print(f"CV Summary (mean+-std) | {data_name} | {mode_label}")

    def _mean_std(values):
        arr = np.asarray(values, dtype=float)
        if arr.size == 0:
            return float('nan'), float('nan')
        finite = arr[~np.isnan(arr)]
        if finite.size == 0:
            return float('nan'), float('nan')
        return float(finite.mean()), float(finite.std())

    rows = []
    for name in summary_models:
        acc_mean, acc_std = _mean_std(cv_results[name]['acc'])
        precision_mean, precision_std = _mean_std(cv_results[name]['precision'])
        recall_mean, recall_std = _mean_std(cv_results[name]['recall'])
        f1_mean, f1_std = _mean_std(cv_results[name]['f1'])
        idx_mean, idx_std = _mean_std(cv_nauck[name]['index'])
        comp_mean, comp_std = _mean_std(cv_nauck[name]['comp'])
        cov_mean, cov_std = _mean_std(cv_nauck[name]['cov'])
        part_mean, part_std = _mean_std(cv_nauck[name]['part'])
        rules_mean, rules_std = _mean_std(cv_nauck[name]['n_rules'])
        feats_mean, feats_std = _mean_std(cv_nauck[name]['n_features'])

        print(
            f"{name:>18} | "
            f"Acc={acc_mean:.4f}+-{acc_std:.4f}, "
            f"Precision={precision_mean:.4f}+-{precision_std:.4f}, "
            f"Recall={recall_mean:.4f}+-{recall_std:.4f}, "
            f"F1={f1_mean:.4f}+-{f1_std:.4f} | "
            f"Nauck idx={idx_mean:.4f}+-{idx_std:.4f}, "
            f"comp={comp_mean:.4f}+-{comp_std:.4f}, "
            f"cov={cov_mean:.4f}+-{cov_std:.4f}, "
            f"part={part_mean:.4f}+-{part_std:.4f}, "
            f"n_rules={rules_mean:.4f}+-{rules_std:.4f}, "
            f"allowed_features={feats_mean:.4f}+-{feats_std:.4f}"
        )

        rows.append({
            'dataset': data_name,
            'mode': mode_label,
            'model': name,
            'acc_mean': acc_mean,
            'acc_std': acc_std,
            'precision_mean': precision_mean,
            'precision_std': precision_std,
            'recall_mean': recall_mean,
            'recall_std': recall_std,
            'f1_mean': f1_mean,
            'f1_std': f1_std,
            'nauck_mean': idx_mean,
            'nauck_std': idx_std,
            'comp_mean': comp_mean,
            'comp_std': comp_std,
            'cov_mean': cov_mean,
            'cov_std': cov_std,
            'part_mean': part_mean,
            'part_std': part_std,
            'n_rules_mean': rules_mean,
            'n_rules_std': rules_std,
            'n_features_mean': feats_mean,
            'n_features_std': feats_std,
        })

    return pd.DataFrame(rows)


In [ ]:
GH_PARAMS_BY_DATASET = {
    'Breast_Cancer_Wisconsin_(Original)': {
        'lr_base': 0.1,
        'lr_residual': 0.1,
        'residual_rules': 11,
        'base_rules': 7,
        'mf_per_feature': 2,
        'epochs_stage1': 80,
        'epochs_stage2': 100,
        'lambda_resid_s2': 0.0,
        'lambda_base_s1': 0.0,
        'weight_decay': 1e-05,
        'base_hard_epochs': 10,
        'residual_hard_epochs': 10,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'rule_init_mode': 'balanced',
        'rule_seed': 0,
        'firing_mode': 'htsk',
        'residual_gate_mode': 'complement',
        'use_input_norm': False,
        'enable_residual_branch': True,
        'random_role_assignment': False,
    },
}

GH_PARAMS = copy.deepcopy(GH_PARAMS_BY_DATASET[DATASET_NAME])
pd.Series(GH_PARAMS).sort_index()


In [ ]:
BCWD_NO_MI_SUMMARY = run_bcwd_cv_dataset_mode(
    data_name=DATASET_NAME,
    gh_params=copy.deepcopy(GH_PARAMS),
    mi_top_k=MI_TOP_K,
    mode_label=MODE_LABEL,
    max_folds=MAX_FOLDS,
)
BCWD_NO_MI_SUMMARY


In [ ]:
summary_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_summary.csv'
BCWD_NO_MI_SUMMARY.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)

summary_view = (
    BCWD_NO_MI_SUMMARY
    .set_index('model')[['acc_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'nauck_mean', 'n_features_mean']]
    .sort_values('acc_mean', ascending=False)
)
summary_view


## Vowel 5-fold performance workflow


# Vowel 5-Fold Experiment

Vowel 데이터셋만 대상으로 5-fold 교차검증을 수행하는 전용 노트북입니다.

- 입력은 `load_vowel_data()`를 그대로 사용하므로, E404 전처리 기준에서 Vowel은 현재 `26`개 feature로 학습됩니다.
- 실행 대상 모델은 `GH-ANFIS`, `ANFIS`, `GA-ANFIS`, `PSO-ANFIS`, `PH-ANFIS(Avg)`, `PH-ANFIS(Stacked)`, `SVM` 입니다.
- fold별 가중치 아티팩트는 `hyper_parameter/cv_weights/Vowel__<mode>/fold_XX/` 아래에 저장됩니다.
- 오래된 Vowel selected-feature 이름(`Speaker_Number_Andrew`, `Sex_Male` 등)도 현재 컬럼명(`Speaker_Number=Andrew`, `Sex`) 기준으로 자동 매칭되도록 helper가 포함되어 있습니다.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()

def _looks_like_project_root(path):
    markers = ["model.py", "data.py", "learning.py", "utils.py"]
    return all((path / marker).exists() for marker in markers)

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E404",
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E404",
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)

os.environ.setdefault("OPENML_DATA_HOME", str(PROJECT_ROOT / "data" / "openml_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
(PROJECT_ROOT / "data" / "openml_cache").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS, TSKANFIS, ParallelHierarchicalTSKANFIS
from utils import set_deterministic, build_loader, load_gh_params, load_ga_params, load_pso_params
from data import *
from learning import *
from gh_eval_utils import *
from interpretability import nauck_index_gh, nauck_index_tsk, nauck_index_parallel_hier_tsk
from sklearn.svm import SVC
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from tqdm import tqdm

print(f"Project root: {PROJECT_ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

set_deterministic(SEED)


In [ ]:
def _metric_average(task_kind):
    return 'weighted' if task_kind == 'multiclass' else 'binary'


def _compute_cls_metrics(y_true, y_pred, task_kind):
    average = _metric_average(task_kind)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
        'recall': recall_score(y_true, y_pred, average=average, zero_division=0),
        'f1': f1_score(y_true, y_pred, average=average, zero_division=0),
    }


def _eval_torch_cls(model, loader, task_kind):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device).float()
            y_batch = y_batch.to(device)
            logits = model(x_batch)

            if task_kind == 'binary':
                if logits.dim() == 1:
                    logits = logits.unsqueeze(1)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long().squeeze(1)
                targets = y_batch.long().squeeze(1) if y_batch.dim() > 1 else y_batch.long()
            else:
                preds = torch.argmax(logits, dim=1)
                targets = y_batch.long()

            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_targets, axis=0)
    return _compute_cls_metrics(y_true, y_pred, task_kind)


def _eval_sklearn_cls(model, X, y, task_kind):
    preds = model.predict(X)
    return _compute_cls_metrics(y, preds, task_kind)


def _eval_gh_detailed_cls(model, loader, task_kind):
    results = {}

    for mode_name in ['base_only', 'residual_only', 'full']:
        model.set_mode(mode_name)
        model.eval()

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for x_batch, y_batch in loader:
                x_batch = x_batch.to(device).float()
                y_batch = y_batch.to(device)
                logits = model(x_batch)

                if task_kind == 'binary':
                    if logits.dim() == 1:
                        logits = logits.unsqueeze(1)
                    probs = torch.sigmoid(logits)
                    preds = (probs >= 0.5).long().squeeze(1)
                    targets = y_batch.long().squeeze(1) if y_batch.dim() > 1 else y_batch.long()
                else:
                    preds = torch.argmax(logits, dim=1)
                    targets = y_batch.long()

                all_preds.append(preds.cpu().numpy())
                all_targets.append(targets.cpu().numpy())

        y_pred = np.concatenate(all_preds, axis=0)
        y_true = np.concatenate(all_targets, axis=0)
        key = 'combined' if mode_name == 'full' else mode_name.replace('_only', '')
        results[key] = _compute_cls_metrics(y_true, y_pred, task_kind)

    model.set_mode('full')
    return results



from pathlib import Path
import copy
import joblib

CV_WEIGHT_ROOT = Path("./hyper_parameter/cv_weights")
_MODEL_FILE_STEM = {
    "GH-ANFIS": "gh_anfis",
    "ANFIS": "anfis",
    "GA-ANFIS": "ga_anfis",
    "PSO-ANFIS": "pso_anfis",
    "PH-ANFIS(Avg)": "ph_anfis_avg",
    "PH-ANFIS(Stacked)": "ph_anfis_stacked",
    "SVM": "svm",
}


def _safe_name(name):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(name))


def _scaler_stats(scaler):
    if scaler is None:
        return None, None
    mean = getattr(scaler, "mean_", None)
    scale = getattr(scaler, "scale_", None)
    return (mean.tolist() if mean is not None else None, scale.tolist() if scale is not None else None)


def save_cv_torch_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    params,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    scaler_mean, scaler_scale = _scaler_stats(scaler)
    stem = _MODEL_FILE_STEM[model_name]
    out_path = fold_dir / f"{stem}.pt"

    state_dict_cpu = {
        k: (v.detach().cpu() if torch.is_tensor(v) else v)
        for k, v in model.state_dict().items()
    }

    payload = {
        "framework": "torch",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "params": copy.deepcopy(params),
        "scaler_mean": scaler_mean,
        "scaler_scale": scaler_scale,
        "state_dict": state_dict_cpu,
        "extra_meta": dict(extra_meta or {}),
    }
    torch.save(payload, out_path)
    return str(out_path)


def save_cv_sklearn_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    stem = _MODEL_FILE_STEM[model_name]
    out_path = fold_dir / f"{stem}.joblib"

    payload = {
        "framework": "sklearn",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "scaler": scaler,
        "model": model,
        "extra_meta": dict(extra_meta or {}),
    }
    joblib.dump(payload, out_path)
    return str(out_path)


def load_cv_artifact(dataset_name, model_name, fold_idx, root_dir=CV_WEIGHT_ROOT, device=device):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    stem = _MODEL_FILE_STEM[model_name]

    if model_name == "SVM":
        payload = joblib.load(fold_dir / f"{stem}.joblib")
        model = payload["model"]
        return model, payload

    payload = torch.load(fold_dir / f"{stem}.pt", map_location=device)
    params = payload.get("params") or {}
    n_features = int(payload["n_features"])
    n_outputs = int(payload["n_outputs"])

    if model_name == "GH-ANFIS":
        model = GH_ANFIS(
            n_features=n_features,
            n_outputs=n_outputs,
            residual_rules=int(params.get("residual_rules", 8)),
            base_rules=int(params.get("base_rules", 4)),
            mf_per_feature=int(params.get("mf_per_feature", 2)),
            device=device,
        ).to(device)
    elif model_name in ("PH-ANFIS(Avg)", "PH-ANFIS(Stacked)"):
        fusion_default = "avg" if model_name == "PH-ANFIS(Avg)" else "stacked"
        fusion = str(params.get("fusion", fusion_default))
        extra_meta = payload.get("extra_meta") or {}
        group_a_idx = list(extra_meta.get("group_a_idx") or [])
        group_b_idx = list(extra_meta.get("group_b_idx") or [])
        if not group_a_idx or not group_b_idx:
            split_seed = int(params.get("split_seed", SEED))
            rng = np.random.default_rng(split_seed)
            order = np.arange(int(n_features), dtype=int)
            rng.shuffle(order)
            cut = int(max(1, n_features // 2))
            if cut >= n_features:
                cut = n_features - 1
            group_a_idx = np.sort(order[:cut]).tolist()
            group_b_idx = np.sort(order[cut:]).tolist()

        model = ParallelHierarchicalTSKANFIS(
            n_inputs=n_features,
            n_outputs=n_outputs,
            group_a_idx=group_a_idx,
            group_b_idx=group_b_idx,
            branch_rules=int(params.get("branch_rules", params.get("n_rules", 12))),
            top_rules=int(params.get("top_rules", max(2, int(params.get("branch_rules", params.get("n_rules", 12))) // 2))),
            fusion=fusion,
            mfs_per_input=int(params.get("mfs_per_input", 3)),
        ).to(device)
    else:
        model = TSKANFIS(
            n_inputs=n_features,
            n_rules=int(params.get("n_rules", 30)),
            n_outputs=n_outputs,
            mfs_per_input=int(params.get("mfs_per_input", 3)),
        ).to(device)

    model.load_state_dict(payload["state_dict"])
    model.eval()
    return model, payload


def build_scaler_from_artifact(payload):
    mean = payload.get("scaler_mean")
    scale = payload.get("scaler_scale")
    if mean is None or scale is None:
        return None
    scaler = StandardScaler()
    scaler.mean_ = np.asarray(mean, dtype=float)
    scaler.scale_ = np.asarray(scale, dtype=float)
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = int(scaler.mean_.shape[0])
    scaler.n_samples_seen_ = 1
    return scaler


def _normalize_feature_key(name):
    text = str(name).strip()
    return "_".join(part for part in ''.join(ch if ch.isalnum() else '_' for ch in text).split('_') if part).lower()


def select_features_df(X, selected, label):
    if not selected:
        return X, list(X.columns) if hasattr(X, "columns") else None
    if not hasattr(X, "columns"):
        return X, None

    available_cols = list(X.columns)
    exact_map = {str(col): col for col in available_cols}
    normalized_map = {}
    for col in available_cols:
        normalized_map.setdefault(_normalize_feature_key(col), []).append(col)

    resolved = []
    missing = []

    for feat in selected:
        feat_str = str(feat)
        if feat_str in exact_map:
            resolved.append(exact_map[feat_str])
            continue

        if feat_str.startswith("x") and feat_str[1:].isdigit():
            idx = int(feat_str[1:])
            if idx < X.shape[1]:
                resolved.append(X.columns[idx])
                continue

        candidates = normalized_map.get(_normalize_feature_key(feat_str), [])
        if len(candidates) == 1:
            resolved.append(candidates[0])
            continue

        missing.append(feat)

    resolved = list(dict.fromkeys(resolved))
    if resolved:
        if missing:
            print(f"[{label}] matched {len(resolved)} selected_features; skipped {len(missing)} unmatched entries")
        return X.loc[:, resolved], list(resolved)

    print(f"[{label}] selected_features not found; using all features")
    return X, list(X.columns)


PH_PARAMS_BY_DATASET = {
    "Vowel": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 18,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Breast_Cancer_Wisconsin_(Original)": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 16,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Spambase": {
        "lr": 0.05,
        "epochs": 80,
        "branch_rules": 20,
        "top_rules": 8,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
    "Gisette": {
        "lr": 0.03,
        "epochs": 70,
        "branch_rules": 14,
        "top_rules": 6,
        "mfs_per_input": 2,
        "weight_decay": 1e-5,
    },
}

PH_VARIANTS = [
    ("PH-ANFIS(Avg)", "avg", 101),
    ("PH-ANFIS(Stacked)", "stacked", 211),
]


In [ ]:
DATASET_NAME = 'Vowel'
MODE_LABEL = 'no_mi'
MI_TOP_K = None
MAX_FOLDS = None  # smoke test가 필요하면 1처럼 줄여서 실행
BATCH_SIZE = 1024
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'vowel_5fold_experiment'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_list = [
    'GH-ANFIS',
    'ANFIS',
    'GA-ANFIS',
    'PSO-ANFIS',
    'PH-ANFIS(Avg)',
    'PH-ANFIS(Stacked)',
    'SVM',
]

VOWEL_x, VOWEL_y, VOWEL_feature_name = load_vowel_data()
VOWEL_x = coerce_numeric_frame(VOWEL_x)
VOWEL_x, VOWEL_y = drop_nan_targets(VOWEL_x, VOWEL_y)
VOWEL_x = VOWEL_x.copy()
VOWEL_y = np.asarray(VOWEL_y, dtype=np.int64)
VOWEL_feature_name = list(VOWEL_x.columns)

print('Vowel shape:', VOWEL_x.shape)
print('First features:', VOWEL_feature_name[:20])
print('Total classes:', len(np.unique(VOWEL_y)))
print('Class counts:')
print(pd.Series(VOWEL_y).value_counts().sort_index())


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd


def _top_mi_columns(X_train_df, y_train, top_k, random_state=SEED):
    if top_k is None:
        return list(X_train_df.columns)
    k = int(max(1, min(int(top_k), X_train_df.shape[1])))
    if k >= X_train_df.shape[1]:
        return list(X_train_df.columns)

    mi = mutual_info_classif(
        X_train_df,
        np.asarray(y_train).astype(int),
        random_state=int(random_state),
    )
    order = np.argsort(mi)[::-1]
    return [X_train_df.columns[i] for i in order[:k]]


def _dataset_bundle(data_name):
    if data_name != DATASET_NAME:
        raise ValueError(f'This notebook is Vowel-only. Received: {data_name}')
    return VOWEL_x, VOWEL_y, VOWEL_feature_name


def _batch_size_for_dataset(data_name):
    return BATCH_SIZE


def run_vowel_cv_dataset_mode(data_name, gh_params, mi_top_k=None, mode_label='no_mi', max_folds=None):
    X_df, y, feature_name = _dataset_bundle(data_name)

    n_classes = len(np.unique(y))
    task = 'binary' if n_classes == 2 else 'multiclass'
    n_out = 1 if task == 'binary' else n_classes
    n_folds = 5
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    ga_params = load_ga_params(data_name, data_name)
    pso_params = load_pso_params(data_name, data_name)
    tsk_params = {'lr': 0.1, 'n_rules': 30, 'epochs': 100, 'selected_features': []}
    ph_params = copy.deepcopy(PH_PARAMS_BY_DATASET[data_name])
    svm_params = {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}

    ga_selected = list(ga_params.get('selected_features') or [])
    pso_selected = list(pso_params.get('selected_features') or [])

    print(f"[{data_name}/{mode_label}] ANFIS selected=core, GA selected={len(ga_selected)}, PSO selected={len(pso_selected)}")

    summary_models = [
        'GH-ANFIS(base)',
        'GH-ANFIS(residual)',
        'GH-ANFIS(full)',
        'ANFIS',
        'GA-ANFIS',
        'PSO-ANFIS',
        'PH-ANFIS(Avg)',
        'PH-ANFIS(Stacked)',
        'SVM',
    ]

    metric_names = ('acc', 'precision', 'recall', 'f1')
    cv_results = {name: {metric: [] for metric in metric_names} for name in summary_models}
    cv_nauck = {
        name: {'index': [], 'comp': [], 'cov': [], 'part': [], 'n_rules': [], 'n_features': []}
        for name in summary_models
    }

    def _push_nauck(name, info):
        cv_nauck[name]['index'].append(info.get('index', np.nan))
        cv_nauck[name]['comp'].append(info.get('comp', np.nan))
        cv_nauck[name]['cov'].append(info.get('cov', np.nan))
        cv_nauck[name]['part'].append(info.get('part', np.nan))
        cv_nauck[name]['n_rules'].append(info.get('n_rules', np.nan))
        cv_nauck[name]['n_features'].append(info.get('n_features', np.nan))

    def _push_metrics(name, metrics):
        for metric in metric_names:
            cv_results[name][metric].append(metrics.get(metric, np.nan))

    saved_cv_artifacts = []

    X_train_df, X_test_df, y_train_tmp, y_test_tmp = train_test_split(
        X_df, y, test_size=0.2, random_state=SEED
    )
    scaler_tmp = StandardScaler().fit(X_train_df)
    X_train_tmp = scaler_tmp.transform(X_train_df)
    X_test_tmp = scaler_tmp.transform(X_test_df)
    print(f"[{data_name}/{mode_label}] Train: {X_train_tmp.shape}, Test: {X_test_tmp.shape}")

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_df, y), 1):
        if max_folds is not None and int(fold_idx) > int(max_folds):
            break
        set_deterministic(SEED)
        print(f"{'='*60}")
        print(f"[{data_name}/{mode_label}] Fold {fold_idx}/{n_folds}")
        print(f"{'='*60}")

        X_train_full, X_val_full = X_df.iloc[train_idx], X_df.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if mi_top_k is None:
            X_train_core = X_train_full
            X_val_core = X_val_full
        else:
            mi_cols = _top_mi_columns(X_train_full, y_train, mi_top_k, random_state=SEED)
            X_train_core = X_train_full.loc[:, mi_cols]
            X_val_core = X_val_full.loc[:, mi_cols]
            if fold_idx == 1:
                print(f"[{data_name}/{mode_label}] MI selected={len(mi_cols)}")

        batch_size = _batch_size_for_dataset(data_name)

        scaler_core = StandardScaler().fit(X_train_core)
        X_train_core_scaled = scaler_core.transform(X_train_core)
        X_val_core_scaled = scaler_core.transform(X_val_core)
        train_loader_core = build_loader(X_train_core_scaled, y_train, batch_size, device, task)
        val_loader_core = build_loader(X_val_core_scaled, y_val, batch_size, device, task)

        X_train_tsk = X_train_core
        X_val_tsk = X_val_core
        tsk_used = list(X_train_core.columns)
        scaler_tsk = StandardScaler().fit(X_train_tsk)
        X_train_tsk_scaled = scaler_tsk.transform(X_train_tsk)
        X_val_tsk_scaled = scaler_tsk.transform(X_val_tsk)
        tsk_train_loader = build_loader(X_train_tsk_scaled, y_train, batch_size, device, task)
        tsk_val_loader = build_loader(X_val_tsk_scaled, y_val, batch_size, device, task)

        X_train_ga, ga_used = select_features_df(X_train_core, ga_selected, 'GA-ANFIS')
        X_val_ga = X_val_core.loc[:, ga_used] if ga_used is not None else X_val_core
        X_train_pso, pso_used = select_features_df(X_train_core, pso_selected, 'PSO-ANFIS')
        X_val_pso = X_val_core.loc[:, pso_used] if pso_used is not None else X_val_core

        scaler_ga = StandardScaler().fit(X_train_ga)
        X_train_ga_scaled = scaler_ga.transform(X_train_ga)
        X_val_ga_scaled = scaler_ga.transform(X_val_ga)
        ga_train_loader = build_loader(X_train_ga_scaled, y_train, batch_size, device, task)
        ga_val_loader = build_loader(X_val_ga_scaled, y_val, batch_size, device, task)

        scaler_pso = StandardScaler().fit(X_train_pso)
        X_train_pso_scaled = scaler_pso.transform(X_train_pso)
        X_val_pso_scaled = scaler_pso.transform(X_val_pso)
        pso_train_loader = build_loader(X_train_pso_scaled, y_train, batch_size, device, task)
        pso_val_loader = build_loader(X_val_pso_scaled, y_val, batch_size, device, task)

        feature_names_core = list(X_train_core.columns)

        gh_model, gh_criterion = fit_gh_anfis(
            gh_params, train_loader_core, X_train_core.shape[1], n_out, task, feature_names_core, device
        )
        tsk_model, _ = fit_tsk_anfis(
            tsk_params, tsk_train_loader, X_train_tsk.shape[1], n_out, task, device
        )
        ga_model, _ = fit_tsk_anfis(
            ga_params, ga_train_loader, X_train_ga.shape[1], n_out, task, device
        )
        pso_model, _ = fit_tsk_anfis(
            pso_params, pso_train_loader, X_train_pso.shape[1], n_out, task, device
        )

        parallel_models = {}
        parallel_meta = {}
        for ph_name, ph_fusion, ph_seed_offset in PH_VARIANTS:
            ph_split_seed = int(SEED + ph_seed_offset + int(fold_idx) * 997 + (0 if mi_top_k is None else int(mi_top_k)))
            ph_model, _, ph_info = fit_parallel_hier_anfis(
                ph_params,
                train_loader_core,
                X_train_core.shape[1],
                n_out,
                task,
                device,
                fusion=ph_fusion,
                split_seed=ph_split_seed,
                feature_names=feature_names_core,
                verbose=False,
            )
            parallel_models[ph_name] = ph_model
            parallel_meta[ph_name] = dict(ph_info or {})
            parallel_meta[ph_name]['split_seed'] = ph_split_seed

        svm_model = SVC(
            C=svm_params['C'],
            gamma=svm_params['gamma'],
            kernel=svm_params['kernel'],
            probability=True,
            random_state=SEED,
        )
        svm_model.fit(X_train_core, y_train)

        full_feature_names = list(X_train_core.columns) if hasattr(X_train_core, 'columns') else [f"x{i}" for i in range(X_train_core.shape[1])]
        tsk_feature_names = list(X_train_tsk.columns)
        ga_feature_names = list(X_train_ga.columns)
        pso_feature_names = list(X_train_pso.columns)

        dataset_artifact_name = f"{data_name}__{mode_label}"
        fold_artifact_paths = {
            'GH-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='GH-ANFIS',
                fold_idx=fold_idx,
                model=gh_model,
                params=gh_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
            ),
            'ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='ANFIS',
                fold_idx=fold_idx,
                model=tsk_model,
                params=tsk_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_tsk.shape[1],
                scaler=scaler_tsk,
                feature_names=tsk_feature_names,
            ),
            'GA-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='GA-ANFIS',
                fold_idx=fold_idx,
                model=ga_model,
                params=ga_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_ga.shape[1],
                scaler=scaler_ga,
                feature_names=ga_feature_names,
            ),
            'PSO-ANFIS': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PSO-ANFIS',
                fold_idx=fold_idx,
                model=pso_model,
                params=pso_params,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_pso.shape[1],
                scaler=scaler_pso,
                feature_names=pso_feature_names,
            ),
            'PH-ANFIS(Avg)': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PH-ANFIS(Avg)',
                fold_idx=fold_idx,
                model=parallel_models['PH-ANFIS(Avg)'],
                params={**ph_params, 'fusion': 'avg', 'split_seed': int((parallel_meta.get('PH-ANFIS(Avg)') or {}).get('split_seed', SEED + 101 + int(fold_idx) * 997))},
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
                extra_meta=parallel_meta.get('PH-ANFIS(Avg)'),
            ),
            'PH-ANFIS(Stacked)': save_cv_torch_artifact(
                dataset_name=dataset_artifact_name,
                model_name='PH-ANFIS(Stacked)',
                fold_idx=fold_idx,
                model=parallel_models['PH-ANFIS(Stacked)'],
                params={**ph_params, 'fusion': 'stacked', 'split_seed': int((parallel_meta.get('PH-ANFIS(Stacked)') or {}).get('split_seed', SEED + 211 + int(fold_idx) * 997))},
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=scaler_core,
                feature_names=full_feature_names,
                extra_meta=parallel_meta.get('PH-ANFIS(Stacked)'),
            ),
            'SVM': save_cv_sklearn_artifact(
                dataset_name=dataset_artifact_name,
                model_name='SVM',
                fold_idx=fold_idx,
                model=svm_model,
                task_kind=task,
                n_outputs=n_out,
                n_features=X_train_core.shape[1],
                scaler=None,
                feature_names=full_feature_names,
            ),
        }
        saved_cv_artifacts.append(fold_artifact_paths)
        print(f"Saved fold artifacts: {Path(next(iter(fold_artifact_paths.values()))).parent}")

        results = _eval_gh_detailed_cls(gh_model, val_loader_core, task)
        gh_base_metrics = results['base']
        gh_resid_metrics = results['residual']
        gh_full_metrics = results['combined']

        tsk_metrics = _eval_torch_cls(tsk_model, tsk_val_loader, task)
        ga_metrics = _eval_torch_cls(ga_model, ga_val_loader, task)
        pso_metrics = _eval_torch_cls(pso_model, pso_val_loader, task)
        ph_avg_metrics = _eval_torch_cls(parallel_models['PH-ANFIS(Avg)'], val_loader_core, task)
        ph_stk_metrics = _eval_torch_cls(parallel_models['PH-ANFIS(Stacked)'], val_loader_core, task)
        svm_metrics = _eval_sklearn_cls(svm_model, X_val_core, y_val, task)

        fold_metrics = {
            'GH-ANFIS': gh_full_metrics,
            'ANFIS': tsk_metrics,
            'GA-ANFIS': ga_metrics,
            'PSO-ANFIS': pso_metrics,
            'PH-ANFIS(Avg)': ph_avg_metrics,
            'PH-ANFIS(Stacked)': ph_stk_metrics,
            'SVM': svm_metrics,
        }

        print('Model results')
        for name in model_list:
            metrics = fold_metrics[name]
            print(
                f"{name:>18} | "
                f"Acc={metrics['acc']:.4f}, "
                f"Precision={metrics['precision']:.4f}, "
                f"Recall={metrics['recall']:.4f}, "
                f"F1={metrics['f1']:.4f}"
            )

        _push_metrics('GH-ANFIS(base)', gh_base_metrics)
        _push_metrics('GH-ANFIS(residual)', gh_resid_metrics)
        _push_metrics('GH-ANFIS(full)', gh_full_metrics)
        _push_metrics('ANFIS', tsk_metrics)
        _push_metrics('GA-ANFIS', ga_metrics)
        _push_metrics('PSO-ANFIS', pso_metrics)
        _push_metrics('PH-ANFIS(Avg)', ph_avg_metrics)
        _push_metrics('PH-ANFIS(Stacked)', ph_stk_metrics)
        _push_metrics('SVM', svm_metrics)

        gh_nauck = nauck_index_gh(gh_model, X_train_core_scaled)
        tsk_nauck = nauck_index_tsk(tsk_model, X_train_tsk_scaled)
        ga_nauck = nauck_index_tsk(ga_model, X_train_ga_scaled)
        pso_nauck = nauck_index_tsk(pso_model, X_train_pso_scaled)
        ph_avg_nauck = nauck_index_parallel_hier_tsk(parallel_models['PH-ANFIS(Avg)'], X_train_core_scaled)
        ph_stk_nauck = nauck_index_parallel_hier_tsk(parallel_models['PH-ANFIS(Stacked)'], X_train_core_scaled)

        base_info = gh_nauck.get('base') or {}
        resid_info = gh_nauck.get('residual') or {}
        overall_info = gh_nauck.get('overall') or {}

        base_rules = float(base_info.get('n_rules', 0) or 0)
        resid_rules = float(resid_info.get('n_rules', 0) or 0)
        total_rules = base_rules + resid_rules
        base_feats = float(base_info.get('n_features', 0) or 0)
        resid_feats = float(resid_info.get('n_features', 0) or 0)
        total_feats = base_feats + resid_feats

        def _weighted_by_count(b_val, b_count, r_val, r_count):
            total = b_count + r_count
            if total <= 0:
                return np.nan
            if np.isnan(b_val):
                b_val = 0.0
            if np.isnan(r_val):
                r_val = 0.0
            return (b_val * b_count + r_val * r_count) / total

        total_antecedents = base_rules * base_feats + resid_rules * resid_feats
        if total_rules > 0 and total_antecedents > 0:
            comp_overall = total_rules / total_antecedents
        else:
            comp_overall = np.nan

        cov_overall = _weighted_by_count(
            base_info.get('cov', np.nan), base_feats,
            resid_info.get('cov', np.nan), resid_feats,
        )
        part_overall = _weighted_by_count(
            base_info.get('part', np.nan), base_feats,
            resid_info.get('part', np.nan), resid_feats,
        )

        overall_info = {
            'index': overall_info.get('index', np.nan),
            'comp': comp_overall,
            'cov': cov_overall,
            'part': part_overall,
            'n_rules': total_rules if total_rules > 0 else overall_info.get('n_rules', np.nan),
            'n_features': total_feats if total_feats > 0 else overall_info.get('n_features', np.nan),
        }

        _push_nauck('GH-ANFIS(base)', base_info)
        _push_nauck('GH-ANFIS(residual)', resid_info)
        _push_nauck('GH-ANFIS(full)', overall_info)
        _push_nauck('ANFIS', tsk_nauck)
        _push_nauck('GA-ANFIS', ga_nauck)
        _push_nauck('PSO-ANFIS', pso_nauck)

        ph_avg_overall = dict((ph_avg_nauck.get('overall') or {}))
        ph_stk_overall = dict((ph_stk_nauck.get('overall') or {}))
        ph_avg_overall.setdefault('n_features', float(X_train_core.shape[1]))
        ph_stk_overall.setdefault('n_features', float(X_train_core.shape[1]))
        _push_nauck('PH-ANFIS(Avg)', ph_avg_overall)
        _push_nauck('PH-ANFIS(Stacked)', ph_stk_overall)

        _push_nauck('SVM', {
            'index': np.nan,
            'comp': np.nan,
            'cov': np.nan,
            'part': np.nan,
            'n_rules': np.nan,
            'n_features': float(X_train_core.shape[1]),
        })

        gh_overall = (gh_nauck.get('overall') or {}).get('index', np.nan)
        gh_base = (gh_nauck.get('base') or {}).get('index', np.nan)
        gh_resid = (gh_nauck.get('residual') or {}).get('index', np.nan)

        print('Nauck (overall)')
        print(f"GH-ANFIS | overall={gh_overall:.4f}, base={gh_base:.4f}, residual={gh_resid:.4f}")
        print(f"ANFIS    | overall={tsk_nauck.get('index', np.nan):.4f}")
        print(f"GA-ANFIS | overall={ga_nauck.get('index', np.nan):.4f}")
        print(f"PSO-ANFIS| overall={pso_nauck.get('index', np.nan):.4f}")
        print(f"PH-ANFIS(Avg)    | overall={(ph_avg_nauck.get('overall') or {}).get('index', np.nan):.4f}")
        print(f"PH-ANFIS(Stacked)| overall={(ph_stk_nauck.get('overall') or {}).get('index', np.nan):.4f}")

    print(f"CV weight root: {CV_WEIGHT_ROOT.resolve()}")
    print(f"Saved fold artifacts: {len(saved_cv_artifacts)} folds")
    print(f"CV Summary (mean+-std) | {data_name} | {mode_label}")

    def _mean_std(values):
        arr = np.asarray(values, dtype=float)
        if arr.size == 0:
            return float('nan'), float('nan')
        finite = arr[~np.isnan(arr)]
        if finite.size == 0:
            return float('nan'), float('nan')
        return float(finite.mean()), float(finite.std())

    rows = []
    for name in summary_models:
        acc_mean, acc_std = _mean_std(cv_results[name]['acc'])
        precision_mean, precision_std = _mean_std(cv_results[name]['precision'])
        recall_mean, recall_std = _mean_std(cv_results[name]['recall'])
        f1_mean, f1_std = _mean_std(cv_results[name]['f1'])
        idx_mean, idx_std = _mean_std(cv_nauck[name]['index'])
        comp_mean, comp_std = _mean_std(cv_nauck[name]['comp'])
        cov_mean, cov_std = _mean_std(cv_nauck[name]['cov'])
        part_mean, part_std = _mean_std(cv_nauck[name]['part'])
        rules_mean, rules_std = _mean_std(cv_nauck[name]['n_rules'])
        feats_mean, feats_std = _mean_std(cv_nauck[name]['n_features'])

        print(
            f"{name:>18} | "
            f"Acc={acc_mean:.4f}+-{acc_std:.4f}, "
            f"Precision={precision_mean:.4f}+-{precision_std:.4f}, "
            f"Recall={recall_mean:.4f}+-{recall_std:.4f}, "
            f"F1={f1_mean:.4f}+-{f1_std:.4f} | "
            f"Nauck idx={idx_mean:.4f}+-{idx_std:.4f}, "
            f"comp={comp_mean:.4f}+-{comp_std:.4f}, "
            f"cov={cov_mean:.4f}+-{cov_std:.4f}, "
            f"part={part_mean:.4f}+-{part_std:.4f}, "
            f"n_rules={rules_mean:.4f}+-{rules_std:.4f}, "
            f"allowed_features={feats_mean:.4f}+-{feats_std:.4f}"
        )

        rows.append({
            'dataset': data_name,
            'mode': mode_label,
            'model': name,
            'acc_mean': acc_mean,
            'acc_std': acc_std,
            'precision_mean': precision_mean,
            'precision_std': precision_std,
            'recall_mean': recall_mean,
            'recall_std': recall_std,
            'f1_mean': f1_mean,
            'f1_std': f1_std,
            'nauck_mean': idx_mean,
            'nauck_std': idx_std,
            'comp_mean': comp_mean,
            'comp_std': comp_std,
            'cov_mean': cov_mean,
            'cov_std': cov_std,
            'part_mean': part_mean,
            'part_std': part_std,
            'n_rules_mean': rules_mean,
            'n_rules_std': rules_std,
            'n_features_mean': feats_mean,
            'n_features_std': feats_std,
        })

    return pd.DataFrame(rows)


In [ ]:
GH_PARAMS = load_gh_params(DATASET_NAME, DATASET_NAME)
GH_PARAMS = copy.deepcopy(GH_PARAMS)
pd.Series(GH_PARAMS).sort_index()


In [ ]:
VOWEL_NO_MI_SUMMARY = run_vowel_cv_dataset_mode(
    data_name=DATASET_NAME,
    gh_params=copy.deepcopy(GH_PARAMS),
    mi_top_k=MI_TOP_K,
    mode_label=MODE_LABEL,
    max_folds=MAX_FOLDS,
)
VOWEL_NO_MI_SUMMARY


In [ ]:
summary_path = OUTPUT_DIR / f'vowel_{MODE_LABEL}_summary.csv'
VOWEL_NO_MI_SUMMARY.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)

summary_view = (
    VOWEL_NO_MI_SUMMARY
    .set_index('model')[['acc_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'nauck_mean', 'n_features_mean']]
    .sort_values('acc_mean', ascending=False)
)
summary_view


## BCWD GH-ANFIS focused workflow


# BCWD GH-ANFIS 5-Fold Training

Breast Cancer Wisconsin (Original) 데이터셋에 대해 `GH-ANFIS`만 5-fold 교차검증으로 학습하는 전용 노트북입니다.

- 입력은 `load_bcwd_data()`를 그대로 사용하므로, E404 전처리 기준에서 BCWD는 one-hot 확장 없이 `9`개 numeric feature로 학습됩니다.
- 하이퍼파라미터는 기본적으로 `hyper_parameter/best_GH-ANFIS_HP.json`의 BCWD 항목을 로드합니다.
- fold별 체크포인트는 `hyper_parameter/cv_weights/Breast_Cancer_Wisconsin__Original___<mode>/fold_XX/gh_anfis.pt`에 저장됩니다.
- 요약 CSV와 best-fold gate 테이블은 `output/bcwd_gh_anfis/` 아래에 저장됩니다.


In [ ]:
from pathlib import Path
import copy
import os
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path):
    markers = ["model.py", "data.py", "learning.py", "utils.py"]
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_E404",
        PROJECT_ROOT / "03_Research" / "GH-ANFIS_exp",
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)

os.environ.setdefault("OPENML_DATA_HOME", str(PROJECT_ROOT / "data" / "openml_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
(PROJECT_ROOT / "data" / "openml_cache").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data import load_bcwd_data, coerce_numeric_frame, drop_nan_targets
from learning import fit_gh_anfis
from gh_eval_utils import evaluate_torch_classification
from utils import set_deterministic, build_loader, load_gh_params

SEED = 42
set_deterministic(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CV_WEIGHT_ROOT = Path("./hyper_parameter/cv_weights")
MODEL_NAME = "GH-ANFIS"


def _safe_name(name):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(name))


def _scaler_stats(scaler):
    if scaler is None:
        return None, None
    mean = getattr(scaler, "mean_", None)
    scale = getattr(scaler, "scale_", None)
    return (mean.tolist() if mean is not None else None, scale.tolist() if scale is not None else None)


def save_cv_torch_artifact(
    dataset_name,
    model_name,
    fold_idx,
    model,
    params,
    task_kind,
    n_outputs,
    n_features,
    scaler=None,
    feature_names=None,
    root_dir=CV_WEIGHT_ROOT,
    extra_meta=None,
):
    dataset_key = _safe_name(dataset_name)
    fold_dir = Path(root_dir) / dataset_key / f"fold_{int(fold_idx):02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    scaler_mean, scaler_scale = _scaler_stats(scaler)
    out_path = fold_dir / "gh_anfis.pt"

    state_dict_cpu = {
        k: (v.detach().cpu() if torch.is_tensor(v) else v)
        for k, v in model.state_dict().items()
    }

    payload = {
        "framework": "torch",
        "dataset": str(dataset_name),
        "dataset_key": dataset_key,
        "model_name": model_name,
        "fold": int(fold_idx),
        "task_kind": str(task_kind),
        "n_outputs": int(n_outputs),
        "n_features": int(n_features),
        "feature_names": list(feature_names) if feature_names is not None else None,
        "params": copy.deepcopy(params),
        "scaler_mean": scaler_mean,
        "scaler_scale": scaler_scale,
        "state_dict": state_dict_cpu,
        "extra_meta": dict(extra_meta or {}),
    }
    torch.save(payload, out_path)
    return str(out_path)


print(f"Project root: {PROJECT_ROOT}")
PROJECT_ROOT, DEVICE


In [ ]:
DATASET_NAME = 'Breast_Cancer_Wisconsin_(Original)'
TASK_KIND = 'binary'
N_OUTPUTS = 1
MODE_LABEL = 'gh_only_5fold'
N_FOLDS = 5
MAX_FOLDS = None  # smoke test가 필요하면 1처럼 줄여서 실행
BATCH_SIZE = 1024
USE_JSON_BEST_PARAMS = True
GH_PARAM_OVERRIDES = {}

OUTPUT_DIR = PROJECT_ROOT / 'output' / 'bcwd_gh_anfis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_df, y, feature_names = load_bcwd_data()
X_df = coerce_numeric_frame(X_df)
X_df, y = drop_nan_targets(X_df, y)
X_df = X_df.copy()
y = np.asarray(y, dtype=np.int64)
feature_names = list(X_df.columns)

print('X shape:', X_df.shape)
print('y shape:', y.shape)
print('n_features:', len(feature_names))
print('class counts:')
print(pd.Series(y).value_counts().sort_index())
print('feature names:', feature_names)


In [ ]:
if USE_JSON_BEST_PARAMS:
    GH_PARAMS = load_gh_params(DATASET_NAME, DATASET_NAME)
else:
    GH_PARAMS = {
        'lr_base': 0.1,
        'lr_residual': 0.005,
        'residual_rules': 11,
        'base_rules': 6,
        'mf_per_feature': 2,
        'epochs_stage1': 80,
        'epochs_stage2': 40,
        'lambda_resid_s2': 0.0,
        'lambda_base_s1': 0.01,
        'weight_decay': 1e-05,
        'base_hard_epochs': 30,
        'residual_hard_epochs': 20,
        'base_mask_threshold': None,
        'residual_mask_threshold': None,
        'lambda_base_hard': 0.0,
        'lambda_resid_hard': 0.0,
        'rule_init_mode': 'balanced',
        'rule_seed': 0,
        'firing_mode': 'htsk',
        'residual_gate_mode': 'complement',
        'use_input_norm': False,
        'enable_residual_branch': True,
        'random_role_assignment': False,
    }

GH_PARAMS = copy.deepcopy(GH_PARAMS)
GH_PARAMS.update(GH_PARAM_OVERRIDES)
GH_PARAMS['rule_seed'] = int(GH_PARAMS.get('rule_seed', SEED))
GH_PARAMS.setdefault('rule_init_mode', 'balanced')
GH_PARAMS.setdefault('firing_mode', 'htsk')
GH_PARAMS.setdefault('residual_gate_mode', 'complement')
GH_PARAMS.setdefault('use_input_norm', False)
GH_PARAMS.setdefault('enable_residual_branch', True)
GH_PARAMS.setdefault('random_role_assignment', False)

pd.Series(GH_PARAMS).sort_index()


In [ ]:
eval_settings = [
    ('base_hard', 'base', 'base_only', False),
    ('base_soft', 'base', 'base_only', True),
    ('residual_hard', 'residual_complement', 'residual_only', False),
    ('full_hard', 'residual_complement', 'full', False),
    ('full_soft', 'residual_complement', 'full', True),
]


def run_bcwd_gh_cv(dataset_name, params, max_folds=None, mode_label='gh_only_5fold'):
    dataset_artifact_name = f"{dataset_name}__{mode_label}"
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_rows = []
    saved_artifact_paths = []
    best_fold_idx = None
    best_fold_full_hard_f1 = -np.inf
    best_fold_results_df = None
    best_fold_gate_df = None

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_df, y), 1):
        if max_folds is not None and int(fold_idx) > int(max_folds):
            break

        set_deterministic(SEED)
        print('=' * 60)
        print(f"[{dataset_name}/{mode_label}] Fold {fold_idx}/{N_FOLDS}")
        print('=' * 60)

        X_train_df = X_df.iloc[train_idx].copy()
        X_val_df = X_df.iloc[val_idx].copy()
        y_train = y[train_idx]
        y_val = y[val_idx]

        scaler = StandardScaler().fit(X_train_df)
        X_train = scaler.transform(X_train_df).astype(np.float32)
        X_val = scaler.transform(X_val_df).astype(np.float32)

        train_loader = build_loader(X_train, y_train, BATCH_SIZE, DEVICE, TASK_KIND)
        val_loader = build_loader(X_val, y_val, BATCH_SIZE, DEVICE, TASK_KIND)

        fold_params = copy.deepcopy(params)
        model, criterion = fit_gh_anfis(
            fold_params,
            train_loader,
            X_train.shape[1],
            N_OUTPUTS,
            TASK_KIND,
            feature_names,
            DEVICE,
            verbose=True,
        )

        fold_results = []
        for label, phase, mode, use_soft_eval in eval_settings:
            model.set_phase(phase)
            model.set_mode(mode)
            loss, acc, f1 = evaluate_torch_classification(
                model,
                val_loader,
                criterion,
                DEVICE,
                TASK_KIND,
                use_soft_eval=use_soft_eval,
            )
            row = {
                'fold': fold_idx,
                'view': label,
                'phase': phase,
                'mode': mode,
                'use_soft_eval': use_soft_eval,
                'loss': loss,
                'acc': acc,
                'f1': f1,
                'train_size': int(X_train.shape[0]),
                'val_size': int(X_val.shape[0]),
            }
            fold_results.append(row)
            fold_rows.append(row)

        base_probs = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
        residual_probs = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
        base_hard = model.base_mask_hard.detach().cpu().numpy()
        residual_hard = model.residual_mask_hard.detach().cpu().numpy()
        residual_effective_hard = residual_hard * (1.0 - base_hard)

        gate_df = pd.DataFrame(
            {
                'fold': fold_idx,
                'feature': feature_names,
                'base_prob': base_probs,
                'base_hard': base_hard,
                'residual_prob': residual_probs,
                'residual_hard': residual_hard,
                'residual_effective_hard': residual_effective_hard,
            }
        )
        gate_df = gate_df.sort_values(
            ['base_hard', 'base_prob', 'residual_effective_hard', 'residual_prob'],
            ascending=[False, False, False, False],
        ).reset_index(drop=True)

        full_hard_row = next(row for row in fold_results if row['view'] == 'full_hard')
        if full_hard_row['f1'] > best_fold_full_hard_f1:
            best_fold_full_hard_f1 = float(full_hard_row['f1'])
            best_fold_idx = int(fold_idx)
            best_fold_results_df = pd.DataFrame(fold_results)
            best_fold_gate_df = gate_df.copy()

        artifact_path = save_cv_torch_artifact(
            dataset_name=dataset_artifact_name,
            model_name=MODEL_NAME,
            fold_idx=fold_idx,
            model=model,
            params=fold_params,
            task_kind=TASK_KIND,
            n_outputs=N_OUTPUTS,
            n_features=X_train.shape[1],
            scaler=scaler,
            feature_names=feature_names,
            extra_meta={
                'seed': SEED,
                'batch_size': BATCH_SIZE,
                'train_size': int(X_train.shape[0]),
                'val_size': int(X_val.shape[0]),
                'positive_ratio_train': float(np.mean(y_train)),
                'positive_ratio_val': float(np.mean(y_val)),
                'fold_results': fold_results,
                'gate_rows': gate_df.to_dict(orient='records'),
            },
        )
        saved_artifact_paths.append(artifact_path)
        print(f"Saved fold artifacts: {Path(artifact_path).parent}")

    fold_results_df = pd.DataFrame(fold_rows)
    summary_rows = []
    for view in [cfg[0] for cfg in eval_settings]:
        view_df = fold_results_df[fold_results_df['view'] == view]
        summary_rows.append({
            'view': view,
            'loss_mean': float(view_df['loss'].mean()),
            'loss_std': float(view_df['loss'].std(ddof=0)),
            'acc_mean': float(view_df['acc'].mean()),
            'acc_std': float(view_df['acc'].std(ddof=0)),
            'f1_mean': float(view_df['f1'].mean()),
            'f1_std': float(view_df['f1'].std(ddof=0)),
        })
    summary_df = pd.DataFrame(summary_rows)

    return {
        'dataset_artifact_name': dataset_artifact_name,
        'saved_artifact_paths': saved_artifact_paths,
        'fold_results_df': fold_results_df,
        'summary_df': summary_df,
        'best_fold_idx': best_fold_idx,
        'best_fold_results_df': best_fold_results_df,
        'best_fold_gate_df': best_fold_gate_df,
    }


In [ ]:
run_output = run_bcwd_gh_cv(
    dataset_name=DATASET_NAME,
    params=copy.deepcopy(GH_PARAMS),
    max_folds=MAX_FOLDS,
    mode_label=MODE_LABEL,
)

DATASET_ARTIFACT_NAME = run_output['dataset_artifact_name']
saved_artifact_paths = run_output['saved_artifact_paths']
fold_results_df = run_output['fold_results_df']
summary_df = run_output['summary_df']
best_fold_idx = run_output['best_fold_idx']
best_fold_results_df = run_output['best_fold_results_df']
best_fold_gate_df = run_output['best_fold_gate_df']

summary_df


In [ ]:
print(f'Best fold by full_hard f1: {best_fold_idx}')
best_fold_results_df


In [ ]:
best_fold_gate_df


In [ ]:
summary_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_summary.csv'
fold_results_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_fold_results.csv'
best_fold_results_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_best_fold_results.csv'
best_fold_gate_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_best_fold_gate.csv'
artifact_paths_path = OUTPUT_DIR / f'bcwd_{MODE_LABEL}_artifact_paths.csv'

summary_df.to_csv(summary_path, index=False)
fold_results_df.to_csv(fold_results_path, index=False)
best_fold_results_df.to_csv(best_fold_results_path, index=False)
best_fold_gate_df.to_csv(best_fold_gate_path, index=False)
pd.DataFrame({'artifact_path': saved_artifact_paths}).to_csv(artifact_paths_path, index=False)

print('Saved summary:', summary_path)
print('Saved fold results:', fold_results_path)
print('Saved best-fold results:', best_fold_results_path)
print('Saved best-fold gate:', best_fold_gate_path)
print('Saved artifact paths:', artifact_paths_path)
